In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

In [ ]:
!unzip -o /content/UA_FloodNet_PyTorch_Prototype.zip -d /content/UA_FloodNet
!find /content/UA_FloodNet -name "smoke_test.py"

In [ ]:
%cd /content/UA_FloodNet/FloodFusion_UA_PyTorch

!pip install -r requirements.txt
!python smoke_test.py

In [ ]:
import os

project = "/content/UA_FloodNet/FloodFusion_UA_PyTorch"

print("Project files:")
for filename in sorted(os.listdir(project)):
    print(filename)

print("\nGPU:")
!nvidia-smi

In [ ]:
%cd /content

!wget -q --show-progress \
  -O OMBRIA-master.zip \
  "https://zenodo.org/records/6777870/files/OMBRIA-master.zip?download=1"

!unzip -q -o OMBRIA-master.zip -d /content/OMBRIA

print("Dataset folders:")
!find /content/OMBRIA -maxdepth 3 -type d | head -60

In [ ]:
import os
from collections import Counter
from PIL import Image
import numpy as np

root = "/content/OMBRIA/OMBRIA-master"

for sensor in ["OmbriaS1", "OmbriaS2"]:
    for split in ["train", "test"]:

        path = os.path.join(root, sensor, split)

        print(f"\n{'='*60}")
        print(f"{sensor} - {split}")
        print(f"{'='*60}")

        files = []

        for directory, _, filenames in os.walk(path):
            for filename in filenames:
                files.append(os.path.join(directory, filename))

        print("Total files:", len(files))

        extensions = Counter(
            os.path.splitext(f)[1].lower()
            for f in files
        )

        print("File formats:", dict(extensions))

        print("\nFirst 10 files:")

        for f in sorted(files)[:10]:
            print(os.path.relpath(f, path))

        print("\nSample image information:")

        for f in sorted(files):
            try:
                with Image.open(f) as img:
                    arr = np.array(img)

                    print("File:", os.path.basename(f))
                    print("Image size:", img.size)
                    print("Image mode:", img.mode)
                    print("Array shape:", arr.shape)
                    print("Data type:", arr.dtype)
                    print("Minimum value:", arr.min())
                    print("Maximum value:", arr.max())
                    break

            except Exception:
                continue

In [ ]:
import os
import re
from collections import Counter

root = "/content/OMBRIA/OMBRIA-master"

def extract_id(filename):
    match = re.search(r'(\d+)(?=\.[^.]+$)', filename)
    return match.group(1) if match else None

for split in ["train", "test"]:
    print(f"\n{'='*60}")
    print(f"DATASET SPLIT: {split.upper()}")
    print(f"{'='*60}")

    sensor_ids = {}

    for sensor in ["OmbriaS1", "OmbriaS2"]:
        path = os.path.join(root, sensor, split)

        sensor_ids[sensor] = {}

        for folder in sorted(os.listdir(path)):
            folder_path = os.path.join(path, folder)

            if not os.path.isdir(folder_path):
                continue

            files = sorted([
                f for f in os.listdir(folder_path)
                if f.lower().endswith(".png")
            ])

            ids = [
                extract_id(f) for f in files
                if extract_id(f) is not None
            ]

            sensor_ids[sensor][folder] = set(ids)

            print(f"\n{sensor}/{folder}")
            print("Number of files:", len(files))
            print("Unique sample IDs:", len(set(ids)))
            print("Example files:", files[:3])
            print("Duplicate IDs:", len(ids) - len(set(ids)))

    common_folders = (
        set(sensor_ids["OmbriaS1"]) &
        set(sensor_ids["OmbriaS2"])
    )

    print("\nShared folders:", sorted(common_folders))

    for folder in sorted(common_folders):
        s1_ids = sensor_ids["OmbriaS1"][folder]
        s2_ids = sensor_ids["OmbriaS2"][folder]

        print(f"\n{folder}:")
        print("Matching S1/S2 IDs:", len(s1_ids & s2_ids))
        print("S1 IDs without S2:", len(s1_ids - s2_ids))
        print("S2 IDs without S1:", len(s2_ids - s1_ids))

In [ ]:
from pathlib import Path
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

root = Path("/content/OMBRIA/OMBRIA-master")

# Check whether training and testing identifiers overlap
def get_ids(sensor, split):
    folder = root / sensor / split / "BEFORE"
    return {
        p.stem.split("_")[-1]
        for p in folder.glob("*.png")
    }

train_ids = get_ids("OmbriaS1", "train")
test_ids = get_ids("OmbriaS1", "test")

print("TRAIN/TEST ID OVERLAP:", len(train_ids & test_ids))

# Examine sample masks from both sensors
for split in ["train", "test"]:
    ids = sorted(get_ids("OmbriaS1", split))
    disagreements = 0
    value_counts = {}

    for sid in ids:
        p1 = root / "OmbriaS1" / split / "MASK" / f"S1_mask_{sid}.png"
        p2 = root / "OmbriaS2" / split / "MASK" / f"S2_mask_{sid}.png"

        with Image.open(p1) as im:
            m1 = np.array(im)

        with Image.open(p2) as im:
            m2 = np.array(im)

        if not np.array_equal(m1, m2):
            disagreements += 1

        for value, count in zip(*np.unique(m1, return_counts=True)):
            value_counts[int(value)] = value_counts.get(int(value), 0) + int(count)

    print(f"\n{split.upper()}")
    print("Number of samples:", len(ids))
    print("Different S1/S2 masks:", disagreements)
    print("S1 mask pixel values:", value_counts)

# Visualize one paired sample
sid = sorted(train_ids)[0]
split = "train"

fig, axes = plt.subplots(2, 3, figsize=(13, 8))

for row, (sensor, prefix) in enumerate([
    ("OmbriaS1", "S1"),
    ("OmbriaS2", "S2")
]):
    for col, (folder, suffix) in enumerate([
        ("BEFORE", "before"),
        ("AFTER", "after"),
        ("MASK", "mask")
    ]):
        path = root / sensor / split / folder / f"{prefix}_{suffix}_{sid}.png"

        with Image.open(path) as im:
            arr = np.array(im)

        axes[row, col].imshow(
            arr,
            cmap="gray" if arr.ndim == 2 else None
        )
        axes[row, col].set_title(f"{sensor} {folder}")
        axes[row, col].axis("off")

plt.suptitle(f"OMBRIA sample {sid}")
plt.tight_layout()
plt.show()

In [ ]:
from pathlib import Path
from PIL import Image
import numpy as np

root = Path("/content/OMBRIA/OMBRIA-master")
output = Path("/content/UA_FloodNet/dataset")

for split in ["train", "val", "test"]:
    (output / split).mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(42)

train_ids = sorted(
    p.stem.split("_")[-1]
    for p in (root / "OmbriaS1/train/BEFORE").glob("*.png")
)

test_ids = sorted(
    p.stem.split("_")[-1]
    for p in (root / "OmbriaS1/test/BEFORE").glob("*.png")
)

shuffled = rng.permutation(train_ids)
n_val = round(len(shuffled) * 0.20)

val_ids = set(shuffled[:n_val])
training_ids = set(shuffled[n_val:])

assert not (training_ids & val_ids)
assert not (set(train_ids) & set(test_ids))

def read_image(sensor, split, stage, sid):
    prefix = "S1" if sensor == "OmbriaS1" else "S2"
    filename = f"{prefix}_{stage.lower()}_{sid}.png"
    path = root / sensor / split / stage / filename

    with Image.open(path) as im:
        if stage != "MASK":
            im = im.convert(
                "L" if sensor == "OmbriaS1" else "RGB"
            )
        arr = np.asarray(im).copy()

    return arr

def save_sample(sid, source_split, target_split):
    s1_pre = read_image(
        "OmbriaS1", source_split, "BEFORE", sid
    )
    s1_post = read_image(
        "OmbriaS1", source_split, "AFTER", sid
    )

    s2_pre = read_image(
        "OmbriaS2", source_split, "BEFORE", sid
    )
    s2_post = read_image(
        "OmbriaS2", source_split, "AFTER", sid
    )

    mask = read_image(
        "OmbriaS1", source_split, "MASK", sid
    )

    assert mask.ndim == 2
    assert set(np.unique(mask).tolist()).issubset({0, 255})

    assert s1_pre.shape == s1_post.shape == mask.shape
    assert s2_pre.shape == s2_post.shape == (*mask.shape, 3)

    sample = {
        "sar_pre": s1_pre[None].astype(np.float32) / 255.0,
        "sar_post": s1_post[None].astype(np.float32) / 255.0,
        "ms_pre": s2_pre.transpose(2, 0, 1).astype(np.float32) / 255.0,
        "ms_post": s2_post.transpose(2, 0, 1).astype(np.float32) / 255.0,
        "mask": (mask[None] == 255).astype(np.float32),
    }

    np.savez_compressed(
        output / target_split / f"sample_{sid}.npz",
        **sample
    )

for sid in sorted(training_ids):
    save_sample(sid, "train", "train")

for sid in sorted(val_ids):
    save_sample(sid, "train", "val")

for sid in test_ids:
    save_sample(sid, "test", "test")

print("DATASET PREPARATION COMPLETED")

for split in ["train", "val", "test"]:
    files = list((output / split).glob("*.npz"))
    print(f"{split}: {len(files)} samples")

example = sorted((output / "train").glob("*.npz"))[0]

with np.load(example) as sample:
    for key in sample.files:
        arr = sample[key]
        print(
            key,
            arr.shape,
            arr.dtype,
            float(arr.min()),
            float(arr.max())
        )

In [ ]:
import sys
import torch
from torch.utils.data import DataLoader

project = "/content/UA_FloodNet/FloodFusion_UA_PyTorch"

if project not in sys.path:
    sys.path.insert(0, project)

from data import NPZFloodDataset

dataset_path = "/content/UA_FloodNet/dataset"

train_dataset = NPZFloodDataset(dataset_path + "/train")
val_dataset = NPZFloodDataset(dataset_path + "/val")
test_dataset = NPZFloodDataset(dataset_path + "/test")

train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=0
)

batch = next(iter(train_loader))

print("DATA LOADER TEST")
print("========================")

for key, tensor in batch.items():
    print(key, tensor.shape, tensor.dtype)

print("\nTraining samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Testing samples:", len(test_dataset))
print("GPU available:", torch.cuda.is_available())

assert batch["sar_pre"].shape == (4, 1, 256, 256)
assert batch["sar_post"].shape == (4, 1, 256, 256)
assert batch["ms_pre"].shape == (4, 3, 256, 256)
assert batch["ms_post"].shape == (4, 3, 256, 256)
assert batch["mask"].shape == (4, 1, 256, 256)

print("\nPASS: Dataset is ready for model training.")

In [ ]:
%cd /content/UA_FloodNet/FloodFusion_UA_PyTorch

!python train.py \
    --train /content/UA_FloodNet/dataset/train \
    --val /content/UA_FloodNet/dataset/val \
    --epochs 2 \
    --batch-size 4 \
    --width 24 \
    --out /content/UA_FloodNet/runs/ua_floodnet_initial.pt

In [ ]:
%cd /content/UA_FloodNet/FloodFusion_UA_PyTorch

!python train.py \
    --train /content/UA_FloodNet/dataset/train \
    --val /content/UA_FloodNet/dataset/val \
    --epochs 50 \
    --batch-size 4 \
    --width 24 \
    --out /content/UA_FloodNet/runs/ua_floodnet_50epochs.pt

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil

drive.mount("/content/drive")

source = Path(
    "/content/UA_FloodNet/runs/ua_floodnet_50epochs.pt"
)

destination = Path(
    "/content/drive/MyDrive/UA_FloodNet/checkpoints/"
)
destination.mkdir(parents=True, exist_ok=True)

assert source.exists(), "Trained checkpoint not found"

saved = destination / source.name
shutil.copy2(source, saved)

print("MODEL SAVED:", saved)
print("Size:", round(saved.stat().st_size / 1024**2, 2), "MB")

In [ ]:
import sys
import torch
from torch.utils.data import DataLoader

project = "/content/UA_FloodNet/FloodFusion_UA_PyTorch"
sys.path.insert(0, project)

from model import UAFloodNet
from data import NPZFloodDataset
from train import scores

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

checkpoint_path = (
    "/content/UA_FloodNet/runs/ua_floodnet_50epochs.pt"
)

checkpoint = torch.load(
    checkpoint_path,
    map_location=device,
    weights_only=True
)

model = UAFloodNet(
    width=checkpoint["width"]
).to(device)

model.load_state_dict(checkpoint["model"])
model.eval()

test_dataset = NPZFloodDataset(
    "/content/UA_FloodNet/dataset/test"
)

test_loader = DataLoader(
    test_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=0
)

results = scores(model, test_loader, device)

print("=" * 45)
print("UA-FLOODNET INDEPENDENT TEST RESULTS")
print("=" * 45)
print("Selected checkpoint epoch:", checkpoint["epoch"])
print("Number of test samples:", len(test_dataset))

for metric, value in results.items():
    print(f"{metric}: {value:.4f}")

print("=" * 45)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

model.eval()

sample = test_dataset[0]

with torch.no_grad():
    inputs = {
        k: sample[k].unsqueeze(0).to(device)
        for k in ["sar_pre", "sar_post", "ms_pre", "ms_post"]
    }
    output = model(**inputs)

    if isinstance(output, dict):
        print("Model output keys:", list(output.keys()))
    else:
        print("Model output type:", type(output))

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import torch

model.eval()

save_dir = "/content/drive/MyDrive/UA_FloodNet/figures"
os.makedirs(save_dir, exist_ok=True)

def to_numpy(x):
    if torch.is_tensor(x):
        return x.detach().cpu().numpy()
    return np.asarray(x)

def visualize_flood(sample_index=0):
    sample = test_dataset[sample_index]

    input_keys = [
        "sar_pre", "sar_post",
        "ms_pre", "ms_post"
    ]

    inputs = {
        key: sample[key].unsqueeze(0).to(device)
        for key in input_keys
    }

    with torch.no_grad():
        output = model(**inputs)

    logits = output["logits"]

    if logits.shape[-2:] != sample["mask"].shape[-2:]:
        logits = torch.nn.functional.interpolate(
            logits,
            size=sample["mask"].shape[-2:],
            mode="bilinear",
            align_corners=False
        )

    probability = torch.sigmoid(logits)[0, 0]
    prediction = (probability >= 0.5).cpu().numpy().astype(np.uint8)

    ground_truth = to_numpy(sample["mask"])[0].astype(np.uint8)

    sar_pre = to_numpy(sample["sar_pre"])[0]
    sar_post = to_numpy(sample["sar_post"])[0]

    ms_pre = np.transpose(to_numpy(sample["ms_pre"]), (1, 2, 0))
    ms_post = np.transpose(to_numpy(sample["ms_post"]), (1, 2, 0))

    # Correct and incorrect pixel classifications
    errors = np.zeros(
        (*ground_truth.shape, 3),
        dtype=np.float32
    )

    true_positive = (prediction == 1) & (ground_truth == 1)
    false_positive = (prediction == 1) & (ground_truth == 0)
    false_negative = (prediction == 0) & (ground_truth == 1)

    errors[true_positive] = [0.1, 0.65, 0.2]
    errors[false_positive] = [0.9, 0.15, 0.15]
    errors[false_negative] = [0.15, 0.35, 0.95]

    intersection = np.logical_and(
        prediction == 1, ground_truth == 1
    ).sum()

    union = np.logical_or(
        prediction == 1, ground_truth == 1
    ).sum()

    iou = intersection / union if union else 1.0

    fig, axes = plt.subplots(2, 4, figsize=(16, 8))

    images = [
        sar_pre,
        sar_post,
        ms_pre,
        ms_post,
        ground_truth,
        prediction,
        to_numpy(probability),
        errors
    ]

    titles = [
        "Pre-flood SAR",
        "Post-flood SAR",
        "Pre-flood multispectral",
        "Post-flood multispectral",
        "Reference flood mask",
        "UA-FloodNet prediction",
        "Flood probability",
        "Classification errors"
    ]

    for ax, img, title in zip(axes.flat, images, titles):
        if img.ndim == 2:
            ax.imshow(
                img,
                cmap="gray" if title != "Flood probability" else "viridis",
                vmin=0,
                vmax=1
            )
        else:
            ax.imshow(np.clip(img, 0, 1))

        ax.set_title(title, fontsize=11)
        ax.axis("off")

    plt.tight_layout()

    filename = os.path.join(
        save_dir,
        f"UA_FloodNet_Test_{sample_index:03d}.png"
    )

    plt.savefig(filename, dpi=300, bbox_inches="tight")
    plt.show()

    print(f"Sample {sample_index}: Flood IoU = {iou:.4f}")
    print("Figure saved:", filename)

# Display the first three test samples
for i in range(3):
    visualize_flood(i)

In [ ]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from pathlib import Path

model.eval()
records = []

with torch.no_grad():
    for i in range(len(test_dataset)):
        sample = test_dataset[i]

        inputs = {
            k: sample[k].unsqueeze(0).to(device)
            for k in ["sar_pre", "sar_post", "ms_pre", "ms_post"]
        }

        output = model(**inputs)

        prob = torch.sigmoid(output["logits"])

        if prob.shape[-2:] != sample["mask"].shape[-2:]:
            prob = torch.nn.functional.interpolate(
                prob,
                size=sample["mask"].shape[-2:],
                mode="bilinear",
                align_corners=False
            )

        pred = (prob[0, 0].cpu().numpy() >= 0.5)
        gt = (sample["mask"][0].cpu().numpy() >= 0.5)

        tp = np.logical_and(pred, gt).sum()
        fp = np.logical_and(pred, ~gt).sum()
        fn = np.logical_and(~pred, gt).sum()
        tn = np.logical_and(~pred, ~gt).sum()

        iou = tp / (tp + fp + fn) if tp + fp + fn else 1.0
        f1 = 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else 1.0
        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        fpr = fp / (fp + tn) if fp + tn else 0.0

        records.append({
            "sample": i,
            "iou": iou,
            "f1": f1,
            "precision": precision,
            "recall": recall,
            "false_positive_rate": fpr,
            "reference_flood_fraction": gt.mean(),
            "predicted_flood_fraction": pred.mean()
        })

df = pd.DataFrame(records)

print("\nTEST SAMPLE SUMMARY")
print(df.describe().round(4).to_string())

print("\n10 WORST SAMPLES")
print(df.nsmallest(10, "iou").round(4).to_string(index=False))

print("\n10 BEST SAMPLES")
print(df.nlargest(10, "iou").round(4).to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 5))

ax.hist(df["iou"], bins=np.linspace(0, 1, 21), edgecolor="black")
ax.set_xlabel("Flood intersection over union")
ax.set_ylabel("Number of test samples")
ax.set_title("UA-FloodNet: Test-sample performance distribution")
ax.set_xlim(0, 1)

plt.tight_layout()
plt.show()

save_path = Path(
    "/content/drive/MyDrive/UA_FloodNet/results"
)
save_path.mkdir(parents=True, exist_ok=True)

df.to_csv(save_path / "test_sample_metrics.csv", index=False)

print("\nResults saved to:", save_path)

In [ ]:
import numpy as np
import torch
import pandas as pd
import matplotlib.pyplot as plt

model.eval()

thresholds = np.arange(0.1, 0.96, 0.05)

def evaluate_thresholds(dataset):
    probabilities = []
    references = []

    with torch.no_grad():
        for i in range(len(dataset)):
            sample = dataset[i]

            inputs = {
                k: sample[k].unsqueeze(0).to(device)
                for k in [
                    "sar_pre", "sar_post",
                    "ms_pre", "ms_post"
                ]
            }

            output = model(**inputs)

            logits = output["logits"]

            if logits.shape[-2:] != sample["mask"].shape[-2:]:
                logits = torch.nn.functional.interpolate(
                    logits,
                    size=sample["mask"].shape[-2:],
                    mode="bilinear",
                    align_corners=False
                )

            prob = torch.sigmoid(logits)[0, 0].cpu().numpy()
            gt = sample["mask"][0].cpu().numpy()

            probabilities.append(prob.flatten())
            references.append(gt.flatten())

    probabilities = np.concatenate(probabilities)
    references = np.concatenate(references) >= 0.5

    results = []

    for threshold in thresholds:
        pred = probabilities >= threshold

        tp = np.logical_and(pred, references).sum()
        fp = np.logical_and(pred, ~references).sum()
        fn = np.logical_and(~pred, references).sum()
        tn = np.logical_and(~pred, ~references).sum()

        results.append({
            "threshold": round(float(threshold), 2),
            "iou": tp / max(tp + fp + fn, 1),
            "f1": 2 * tp / max(2 * tp + fp + fn, 1),
            "precision": tp / max(tp + fp, 1),
            "recall": tp / max(tp + fn, 1),
            "fpr": fp / max(fp + tn, 1)
        })

    return pd.DataFrame(results)

validation_results = evaluate_thresholds(val_dataset)

best_row = validation_results.loc[
    validation_results["iou"].idxmax()
]

best_threshold = float(best_row["threshold"])

print("VALIDATION THRESHOLD ANALYSIS")
print(validation_results.round(4).to_string(index=False))

print("\nBEST VALIDATION THRESHOLD:", best_threshold)
print("Best validation IoU:", round(best_row["iou"], 4))

fig, ax = plt.subplots(figsize=(9, 5))

for metric in ["iou", "f1", "precision", "recall"]:
    ax.plot(
        validation_results["threshold"],
        validation_results[metric],
        marker="o",
        label=metric.upper()
    )

ax.axvline(
    best_threshold,
    linestyle="--",
    label="Selected threshold"
)

ax.set_xlabel("Classification probability threshold")
ax.set_ylabel("Performance")
ax.set_title("UA-FloodNet validation threshold sensitivity")
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import torch
from pathlib import Path

model.eval()
threshold = 0.70

tp = fp = fn = tn = 0
sample_results = []

with torch.no_grad():
    for i in range(len(test_dataset)):
        sample = test_dataset[i]

        inputs = {
            k: sample[k].unsqueeze(0).to(device)
            for k in ["sar_pre", "sar_post", "ms_pre", "ms_post"]
        }

        output = model(**inputs)
        logits = output["logits"]

        if logits.shape[-2:] != sample["mask"].shape[-2:]:
            logits = torch.nn.functional.interpolate(
                logits,
                size=sample["mask"].shape[-2:],
                mode="bilinear",
                align_corners=False
            )

        probabilities = torch.sigmoid(logits)[0, 0].cpu().numpy()
        prediction = probabilities >= threshold
        reference = sample["mask"][0].cpu().numpy() >= 0.5

        a = int(np.logical_and(prediction, reference).sum())
        b = int(np.logical_and(prediction, ~reference).sum())
        c = int(np.logical_and(~prediction, reference).sum())
        d = int(np.logical_and(~prediction, ~reference).sum())

        tp += a
        fp += b
        fn += c
        tn += d

        sample_results.append({
            "sample": i,
            "iou": a / max(a + b + c, 1),
            "precision": a / max(a + b, 1),
            "recall": a / max(a + c, 1)
        })

results = {
    "flood_iou": tp / max(tp + fp + fn, 1),
    "flood_f1": 2 * tp / max(2 * tp + fp + fn, 1),
    "precision": tp / max(tp + fp, 1),
    "recall": tp / max(tp + fn, 1),
    "accuracy": (tp + tn) / max(tp + tn + fp + fn, 1),
    "false_positive_rate": fp / max(fp + tn, 1)
}

print("UA-FLOODNET TEST RESULTS")
print("Selected threshold:", threshold)
print("Test samples:", len(test_dataset))

for metric, value in results.items():
    print(f"{metric}: {value:.4f}")

df_threshold = pd.DataFrame(sample_results)

print("\nMean sample IoU:", round(df_threshold["iou"].mean(), 4))
print("\nWorst five test samples:")
print(df_threshold.nsmallest(5, "iou").round(4).to_string(index=False))

save_dir = Path("/content/drive/MyDrive/UA_FloodNet/results")
save_dir.mkdir(parents=True, exist_ok=True)

pd.DataFrame([{
    "threshold": threshold,
    **results
}]).to_csv(save_dir / "test_threshold_070.csv", index=False)

df_threshold.to_csv(
    save_dir / "test_samples_threshold_070.csv",
    index=False
)

print("\nResults saved to Google Drive.")

In [ ]:
from pathlib import Path

path = Path(
    "/content/UA_FloodNet/FloodFusion_UA_PyTorch/model.py"
)

print(path.read_text())

In [ ]:
from pathlib import Path
import zipfile

archive = Path("/content/UA_FloodNet_Ablations.zip")
destination = Path("/content/UA_FloodNet/ablations_code")

destination.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(archive) as z:
    z.extractall(destination)

print("ABLATION SCRIPTS")
for file in sorted(destination.iterdir()):
    print(file.name)

In [ ]:
%cd /content/UA_FloodNet/ablations_code

!python train_ablation.py \
    --variant base \
    --train /content/UA_FloodNet/dataset/train \
    --val /content/UA_FloodNet/dataset/val \
    --epochs 50 \
    --batch-size 4 \
    --width 24 \
    --seed 42 \
    --out /content/drive/MyDrive/UA_FloodNet/ablations/base_seed42.pt

In [ ]:
%cd /content/UA_FloodNet/ablations_code

!python train_ablation.py \
    --variant align \
    --train /content/UA_FloodNet/dataset/train \
    --val /content/UA_FloodNet/dataset/val \
    --epochs 50 \
    --batch-size 4 \
    --width 24 \
    --seed 42 \
    --out /content/drive/MyDrive/UA_FloodNet/ablations/align_seed42.pt

In [ ]:
%cd /content/UA_FloodNet/ablations_code

!python train_ablation.py \
    --variant alignment \
    --train /content/UA_FloodNet/dataset/train \
    --val /content/UA_FloodNet/dataset/val \
    --epochs 50 \
    --batch-size 4 \
    --width 24 \
    --seed 42 \
    --out /content/drive/MyDrive/UA_FloodNet/ablations/align_seed42.pt

In [ ]:
%cd /content/UA_FloodNet/ablations_code

!python train_ablation.py \
    --variant reliability \
    --train /content/UA_FloodNet/dataset/train \
    --val /content/UA_FloodNet/dataset/val \
    --epochs 50 \
    --batch-size 4 \
    --width 24 \
    --seed 42 \
    --out /content/drive/MyDrive/UA_FloodNet/ablations/reliability_seed42.pt

In [ ]:
%cd /content/UA_FloodNet/ablations_code

!python train_ablation.py \
    --variant full \
    --train /content/UA_FloodNet/dataset/train \
    --val /content/UA_FloodNet/dataset/val \
    --epochs 50 \
    --batch-size 4 \
    --width 24 \
    --seed 42 \
    --out /content/drive/MyDrive/UA_FloodNet/ablations/full_seed42.pt

In [ ]:
from pathlib import Path
import torch

folder = Path(
    "/content/drive/MyDrive/UA_FloodNet/ablations"
)

checkpoints = {
    "Base": "base_seed42.pt",
    "Alignment": "align_seed42.pt",
    "Reliability": "reliability_seed42.pt",
    "Full": "full_seed42.pt"
}

print("ABLATION CHECKPOINT VERIFICATION")
print("=" * 55)

for name, filename in checkpoints.items():
    path = folder / filename

    if not path.exists():
        print(name, "MISSING")
        continue

    ckpt = torch.load(
        path,
        map_location="cpu",
        weights_only=True
    )

    print(f"\n{name}")
    print("File:", filename)
    print("Size:", round(path.stat().st_size / 1024**2, 2), "MB")
    print("Checkpoint keys:", list(ckpt.keys()))

    for key in ["variant", "epoch", "width"]:
        if key in ckpt:
            print(f"{key}:", ckpt[key])

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from pathlib import Path
from torch.utils.data import DataLoader

code_dir = "/content/UA_FloodNet/ablations_code"
if code_dir not in sys.path:
    sys.path.insert(0, code_dir)

from ablation_model import UAFloodNet
from data import NPZFloodDataset

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

checkpoint_dir = Path(
    "/content/drive/MyDrive/UA_FloodNet/ablations"
)

variants = {
    "Base": "base_seed42.pt",
    "Alignment": "align_seed42.pt",
    "Reliability": "reliability_seed42.pt",
    "Full": "full_seed42.pt"
}

# Use validation data for this exploratory robustness experiment.
dataset = NPZFloodDataset(
    "/content/UA_FloodNet/dataset/val"
)
loader = DataLoader(
    dataset, batch_size=4, shuffle=False, num_workers=0
)

shifts = [0, 1, 2, 4, 8]

def shift_right(x, pixels):
    """Translate the MS tensor horizontally with zero padding.
    No wraparound and no change to SAR or reference masks.
    """
    if pixels == 0:
        return x

    result = torch.zeros_like(x)
    result[..., pixels:] = x[..., :-pixels]
    return result

def evaluate(model, shift):
    model.eval()
    tp = fp = fn = tn = 0

    with torch.inference_mode():
        for batch in loader:
            sar_pre = batch["sar_pre"].to(device)
            sar_post = batch["sar_post"].to(device)
            ms_pre = batch["ms_pre"].to(device)
            ms_post = batch["ms_post"].to(device)
            truth = batch["mask"].to(device) >= 0.5

            # Identical shift for both MS time points
            ms_pre = shift_right(ms_pre, shift)
            ms_post = shift_right(ms_post, shift)

            output = model(
                sar_pre=sar_pre,
                sar_post=sar_post,
                ms_pre=ms_pre,
                ms_post=ms_post
            )

            logits = output["logits"]

            if logits.shape[-2:] != truth.shape[-2:]:
                logits = torch.nn.functional.interpolate(
                    logits,
                    size=truth.shape[-2:],
                    mode="bilinear",
                    align_corners=False
                )

            prediction = torch.sigmoid(logits) >= 0.5

            tp += int((prediction & truth).sum().item())
            fp += int((prediction & ~truth).sum().item())
            fn += int((~prediction & truth).sum().item())
            tn += int((~prediction & ~truth).sum().item())

    return {
        "iou": tp / max(tp + fp + fn, 1),
        "f1": 2 * tp / max(2 * tp + fp + fn, 1),
        "precision": tp / max(tp + fp, 1),
        "recall": tp / max(tp + fn, 1),
        "accuracy": (tp + tn) / max(tp + fp + fn + tn, 1)
    }

records = []

for variant, filename in variants.items():
    path = checkpoint_dir / filename

    checkpoint = torch.load(
        path, map_location="cpu", weights_only=True
    )

    model = UAFloodNet(
        width=checkpoint["width"],
        use_alignment=checkpoint["use_alignment"],
        use_reliability=checkpoint["use_reliability"]
    ).to(device)

    model.load_state_dict(checkpoint["model"])
    model.eval()

    print(f"\nEvaluating {variant}")

    for shift in shifts:
        metrics = evaluate(model, shift)

        records.append({
            "variant": variant,
            "shift_pixels": shift,
            **metrics
        })

        print(
            f"Shift={shift:2d} px | "
            f"IoU={metrics['iou']:.4f} | "
            f"F1={metrics['f1']:.4f} | "
            f"Precision={metrics['precision']:.4f} | "
            f"Recall={metrics['recall']:.4f}"
        )

    del model
    if device.type == "cuda":
        torch.cuda.empty_cache()

results = pd.DataFrame(records)

print("\nSPATIAL MISREGISTRATION COMPARISON")
comparison = results.pivot(
    index="shift_pixels",
    columns="variant",
    values="iou"
)
print(comparison.round(4).to_string())

# Relative degradation from each model's own zero-shift IoU
relative_drop = (
    100 * (comparison.loc[0] - comparison)
    / comparison.loc[0]
)

print("\nRELATIVE IoU DEGRADATION (%)")
print(relative_drop.round(2).to_string())

# Visualization
fig, ax = plt.subplots(figsize=(9, 5))

for variant in variants:
    ax.plot(
        comparison.index,
        comparison[variant],
        marker="o",
        linewidth=2,
        label=variant
    )

ax.set_xlabel("Artificial Sentinel-2 horizontal shift (pixels)")
ax.set_ylabel("Validation Flood IoU")
ax.set_title("UA-FloodNet robustness to spatial misregistration")
ax.set_xticks(shifts)
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()

save_dir = Path(
    "/content/drive/MyDrive/UA_FloodNet/robustness"
)
save_dir.mkdir(parents=True, exist_ok=True)

results.to_csv(
    save_dir / "misregistration_results.csv",
    index=False
)

fig.savefig(
    save_dir / "misregistration_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("\nResults saved:", save_dir)

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from ablation_model import UAFloodNet

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

checkpoint_path = (
    "/content/drive/MyDrive/UA_FloodNet/ablations/full_seed42.pt"
)

ckpt = torch.load(
    checkpoint_path,
    map_location=device,
    weights_only=True
)

model = UAFloodNet(
    width=ckpt["width"],
    use_alignment=ckpt["use_alignment"],
    use_reliability=ckpt["use_reliability"]
).to(device)

model.load_state_dict(ckpt["model"])
model.eval()

sample = val_dataset[0]

inputs = {
    k: sample[k].unsqueeze(0).to(device)
    for k in ["sar_pre", "sar_post", "ms_pre", "ms_post"]
}

with torch.no_grad():
    output = model(**inputs)

print("Number of alignment levels:", len(output["flows"]))

fig, axes = plt.subplots(
    len(output["flows"]), 2,
    figsize=(10, 12)
)

for level, pair in enumerate(output["flows"]):
    for time, flow in enumerate(pair):
        flow_np = flow[0].detach().cpu().numpy()

        magnitude = np.sqrt(
            flow_np[0] ** 2 + flow_np[1] ** 2
        )

        axes[level, time].imshow(
            magnitude,
            cmap="viridis"
        )

        axes[level, time].set_title(
            f"Level {level + 1}, Time {time + 1}"
        )
        axes[level, time].axis("off")

        print(
            f"Level {level + 1}, Time {time + 1}:",
            f"Mean magnitude = {magnitude.mean():.4f}",
            f"Max magnitude = {magnitude.max():.4f}"
        )

plt.suptitle("UA-FloodNet learned displacement magnitude")
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 4, figsize=(16, 11))

for level, pair in enumerate(output["flows"]):
    for time, flow in enumerate(pair):
        f = flow[0].detach().cpu().numpy()

        dx = f[0]
        dy = f[1]

        for component, data in enumerate([dx, dy]):
            col = time * 2 + component

            limit = 4.0 / (2 ** level)

            im = axes[level, col].imshow(
                data,
                cmap="RdBu_r",
                vmin=-limit,
                vmax=limit
            )

            axes[level, col].set_title(
                f"Level {level+1}, T{time+1}, "
                f"{'dx' if component == 0 else 'dy'}"
            )

            axes[level, col].axis("off")

            print(
                f"Level {level+1}, T{time+1}, "
                f"{'dx' if component == 0 else 'dy'}:",
                f"mean={data.mean():.4f}, "
                f"min={data.min():.4f}, "
                f"max={data.max():.4f}"
            )

plt.tight_layout()
plt.show()

In [ ]:
!unzip -oq /content/UA_FloodNet_V2.zip -d /content/UA_FloodNet_V2

%cd /content/UA_FloodNet_V2

!python smoke_test_v2.py

In [ ]:
!python train_v2.py \
    --train /content/UA_FloodNet/dataset/train \
    --val /content/UA_FloodNet/dataset/val \
    --epochs 5 \
    --batch-size 4 \
    --width 24 \
    --seed 42 \
    --max-train-shift 4 \
    --shift-prob 0.75 \
    --mag-weight 0.002 \
    --epe-weight 0.1 \
    --out /content/drive/MyDrive/UA_FloodNet/v2/v2_pilot.pt

In [ ]:
import torch
import numpy as np
from pathlib import Path

checkpoint_path = Path(
    "/content/drive/MyDrive/UA_FloodNet/v2/v2_pilot.pt"
)

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=True
)

print("V2 CHECKPOINT VERIFICATION")
print("=" * 50)
print("Keys:", list(checkpoint.keys()))

for key in [
    "epoch", "width", "variant",
    "use_alignment", "use_reliability"
]:
    if key in checkpoint:
        print(key, ":", checkpoint[key])

print("\nSaved model parameter groups:")
for key in list(checkpoint["model"].keys())[:15]:
    print(key)

In [ ]:
from pathlib import Path
import ast

v2_folder = Path("/content/UA_FloodNet_V2")

print("PYTHON FILES")
print("=" * 50)

for path in sorted(v2_folder.glob("*.py")):
    print(path.name)

print("\nMODEL CLASS DEFINITIONS")
print("=" * 50)

for path in sorted(v2_folder.glob("*.py")):
    source = path.read_text()
    tree = ast.parse(source)

    for node in tree.body:
        if isinstance(node, ast.ClassDef):
            print(f"{path.name}: {node.name}")

print("\nTRAINING SCRIPT MODEL IMPORTS")
print("=" * 50)

train_path = v2_folder / "train_v2.py"

for line in train_path.read_text().splitlines():
    if (
        "import" in line
        or "UAFloodNet(" in line
        or "model =" in line
    ):
        print(line)

In [ ]:
import sys
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

v2_dir = "/content/UA_FloodNet_V2"
sys.path.insert(0, v2_dir)

from model_v2 import UAFloodNetV2
from data import NPZFloodDataset
from registration import (
    translate_zero,
    displacement_diagnostics
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Load best V2 checkpoint
checkpoint = torch.load(
    "/content/drive/MyDrive/UA_FloodNet/v2/v2_pilot.pt",
    map_location="cpu",
    weights_only=True
)

model_v2 = UAFloodNetV2(
    width=checkpoint["width"]
).to(device)

model_v2.load_state_dict(checkpoint["model"])
model_v2.eval()

# Load validation sample
dataset_v2 = NPZFloodDataset(
    "/content/UA_FloodNet/dataset/val"
)

sample = dataset_v2[0]

inputs = {
    key: sample[key].unsqueeze(0).to(device)
    for key in ["sar_pre", "sar_post", "ms_pre", "ms_post"]
}

with torch.inference_mode():
    output_v2 = model_v2(**inputs)

# Numerical displacement diagnostics
report = displacement_diagnostics(output_v2)
df = pd.DataFrame(report)

print("UA-FLOODNET V2 DISPLACEMENT DIAGNOSTICS")
print("=" * 65)
print(df.round(4).to_string(index=False))

# Visualise displacement components and magnitude
fig, axes = plt.subplots(3, 6, figsize=(18, 10))

for level, pair in enumerate(output_v2["flows"]):
    limit = 4.0 / (2 ** level)

    for time, flow in enumerate(pair):
        f = flow[0].detach().cpu().numpy()

        dx, dy = f[0], f[1]
        magnitude = np.sqrt(dx**2 + dy**2)

        col = time * 3

        axes[level, col].imshow(
            dx, cmap="RdBu_r",
            vmin=-limit, vmax=limit
        )

        axes[level, col + 1].imshow(
            dy, cmap="RdBu_r",
            vmin=-limit, vmax=limit
        )

        axes[level, col + 2].imshow(
            magnitude, cmap="viridis",
            vmin=0, vmax=np.sqrt(2)*limit
        )

        for j, label in enumerate(["dx", "dy", "Magnitude"]):
            axes[level, col + j].set_title(
                f"Level {level+1}, T{time+1}, {label}"
            )
            axes[level, col + j].axis("off")

plt.tight_layout()

save_dir = Path(
    "/content/drive/MyDrive/UA_FloodNet/v2/diagnostics"
)
save_dir.mkdir(parents=True, exist_ok=True)

fig.savefig(
    save_dir / "v2_displacement_fields.png",
    dpi=300,
    bbox_inches="tight"
)

df.to_csv(
    save_dir / "v2_displacement_statistics.csv",
    index=False
)

plt.show()

print("\nDiagnostics saved to:", save_dir)

In [ ]:
import torch
import numpy as np
import pandas as pd

model_v2.eval()

sample = dataset_v2[0]

inputs = {
    k: sample[k].unsqueeze(0).to(device)
    for k in ["sar_pre", "sar_post", "ms_pre", "ms_post"]
}

shifts = [0, 1, 2, 4]
records = []

with torch.inference_mode():

    baseline = model_v2(**inputs)

    for shift in shifts:

        shifted_inputs = dict(inputs)

        shifted_inputs["ms_pre"] = translate_zero(
            inputs["ms_pre"], dx=shift, dy=0
        )

        shifted_inputs["ms_post"] = translate_zero(
            inputs["ms_post"], dx=shift, dy=0
        )

        output = model_v2(**shifted_inputs)

        for level, (flow_pair, base_pair) in enumerate(
            zip(output["flows"], baseline["flows"])
        ):

            scale = 2 ** level

            for time in range(2):

                flow = flow_pair[time]
                base_flow = base_pair[time]

                # Convert feature-grid shifts to image-pixel units
                delta = (flow - base_flow) * scale

                # Exclude artificial boundary areas
                margin = max(shift + 4, 8)
                margin = max(1, margin // scale)

                h, w = delta.shape[-2:]

                if 2 * margin < min(h, w):
                    delta = delta[
                        :, :,
                        margin:h-margin,
                        margin:w-margin
                    ]

                mean_dx = delta[:, 0].mean().item()
                mean_dy = delta[:, 1].mean().item()

                records.append({
                    "shift_pixels": shift,
                    "level": level + 1,
                    "time": time + 1,
                    "delta_dx": mean_dx,
                    "delta_dy": mean_dy,
                    "expected_dx": shift
                })

df_shift = pd.DataFrame(records)

print("V2 SYNTHETIC DISPLACEMENT RESPONSE")
print("=" * 65)
print(df_shift.round(4).to_string(index=False))

In [ ]:
from pathlib import Path

folder = Path("/content/UA_FloodNet_V2")

for filename in ["registration.py", "train_v2.py", "model_v2.py"]:
    path = folder / filename

    print("\n" + "=" * 75)
    print(filename)
    print("=" * 75)

    print(path.read_text())

In [ ]:
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from registration import translate_zero

model_v2.eval()

loader = DataLoader(
    dataset_v2,
    batch_size=4,
    shuffle=False,
    num_workers=0
)

shifts = [(0, 0), (1, 0), (2, 0),
          (4, 0), (-2, 0), (0, 2), (0, -2)]

records = []

with torch.inference_mode():

    for dx, dy in shifts:
        errors = [[] for _ in range(3)]
        mean_dx = [[] for _ in range(3)]
        mean_dy = [[] for _ in range(3)]

        for batch in loader:

            inputs = {
                k: v.to(device)
                for k, v in batch.items()
                if k in [
                    "sar_pre", "sar_post",
                    "ms_pre", "ms_post"
                ]
            }

            inputs["ms_pre"] = translate_zero(
                inputs["ms_pre"], dx, dy
            )
            inputs["ms_post"] = translate_zero(
                inputs["ms_post"], dx, dy
            )

            output = model_v2(**inputs)

            for level, pair in enumerate(output["flows"]):
                scale = 2 ** level

                for flow in pair:
                    # Convert predictions to input-image pixels
                    flow_pixels = flow * scale

                    h, w = flow_pixels.shape[-2:]

                    # Exclude areas affected by zero padding
                    margin = max(
                        2, int(np.ceil((max(abs(dx), abs(dy)) + 4) / scale))
                    )

                    valid = flow_pixels[
                        :, :, margin:h-margin, margin:w-margin
                    ]

                    target = torch.tensor(
                        [dx, dy],
                        dtype=valid.dtype,
                        device=valid.device
                    ).view(1, 2, 1, 1)

                    epe = torch.linalg.vector_norm(
                        valid - target,
                        dim=1
                    ).mean()

                    errors[level].append(epe.item())
                    mean_dx[level].append(valid[:, 0].mean().item())
                    mean_dy[level].append(valid[:, 1].mean().item())

        for level in range(3):
            records.append({
                "shift_dx": dx,
                "shift_dy": dy,
                "level": level + 1,
                "mean_pred_dx": np.mean(mean_dx[level]),
                "mean_pred_dy": np.mean(mean_dy[level]),
                "endpoint_error": np.mean(errors[level])
            })

df_registration = pd.DataFrame(records)

print("UA-FLOODNET V2 REGISTRATION ACCURACY")
print("=" * 70)
print(df_registration.round(4).to_string(index=False))

save_dir = (
    "/content/drive/MyDrive/UA_FloodNet/v2/diagnostics/"
)

df_registration.to_csv(
    save_dir + "registration_endpoint_error.csv",
    index=False
)

In [ ]:
import shutil

shutil.copy2(
    "/content/train_registration_v3.py",
    "/content/UA_FloodNet_V2/train_registration_v3.py"
)

print("V3 registration script installed successfully.")

In [ ]:
%cd /content/UA_FloodNet_V2

!python train_registration_v3.py \
    --train /content/UA_FloodNet/dataset/train \
    --val /content/UA_FloodNet/dataset/val \
    --epochs 5 \
    --batch-size 4 \
    --width 24 \
    --max-shift 4 \
    --lr 0.0003 \
    --seed 42 \
    --out /content/drive/MyDrive/UA_FloodNet/v3/registration_pilot.pt

In [ ]:
!python /content/UA_FloodNet_V2/train_registration_v3.py --help

In [ ]:
%cd /content/UA_FloodNet_V2

!python train_registration_v3.py \
    --train /content/UA_FloodNet/dataset/train \
    --val /content/UA_FloodNet/dataset/val \
    --epochs 20 \
    --batch-size 4 \
    --width 24 \
    --max-shift 4 \
    --lr 0.0003 \
    --seed 42 \
    --out /content/drive/MyDrive/UA_FloodNet/v3/registration_20epochs.pt

In [ ]:
import torch
from pathlib import Path

path = Path(
    "/content/drive/MyDrive/UA_FloodNet/v3/registration_20epochs.pt"
)

checkpoint = torch.load(
    path,
    map_location="cpu",
    weights_only=True
)

print("V3 CHECKPOINT VERIFICATION")
print("=" * 55)
print("Checkpoint exists:", path.exists())
print("File size:", round(path.stat().st_size / 1024**2, 2), "MB")
print("Checkpoint keys:", list(checkpoint.keys()))

for key in [
    "epoch", "width", "seed",
    "best_epe", "val_epe", "max_shift"
]:
    if key in checkpoint:
        print(f"{key}: {checkpoint[key]}")

In [ ]:
import sys
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from torch.utils.data import DataLoader

sys.path.insert(0, "/content/UA_FloodNet_V2")

from model_v2 import UAFloodNetV2
from registration import translate_zero
from data import NPZFloodDataset

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

checkpoint = torch.load(
    "/content/drive/MyDrive/UA_FloodNet/v3/registration_20epochs.pt",
    map_location="cpu",
    weights_only=True
)

model_v3 = UAFloodNetV2(
    width=checkpoint["width"]
).to(device)

model_v3.load_state_dict(checkpoint["model"])
model_v3.eval()

val_dataset = NPZFloodDataset(
    "/content/UA_FloodNet/dataset/val"
)

val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=0
)

# Fixed displacement experiments
translations = [
    (0, 0),
    (1, 0),
    (2, 0),
    (4, 0),
    (-1, 0),
    (-2, 0),
    (-4, 0),
    (0, 2),
    (0, -2),
    (2, 2),
    (-2, -2)
]

records = []

with torch.inference_mode():

    for dx, dy in translations:

        for level in range(3):
            total_epe = 0.0
            total_dx = 0.0
            total_dy = 0.0
            total_pixels = 0

            for batch in val_loader:

                inputs = {
                    k: v.to(device)
                    for k, v in batch.items()
                    if k in [
                        "sar_pre", "sar_post",
                        "ms_pre", "ms_post"
                    ]
                }

                inputs["ms_pre"] = translate_zero(
                    inputs["ms_pre"], dx, dy
                )

                inputs["ms_post"] = translate_zero(
                    inputs["ms_post"], dx, dy
                )

                output = model_v3(**inputs)

                for flow in output["flows"][level]:

                    scale = 2 ** level
                    flow_pixels = flow * scale

                    _, _, h, w = flow_pixels.shape

                    margin = max(
                        2,
                        int(np.ceil(
                            (max(abs(dx), abs(dy)) + 4) / scale
                        ))
                    )

                    cropped = flow_pixels[
                        :, :,
                        margin:h-margin,
                        margin:w-margin
                    ]

                    target = torch.tensor(
                        [dx, dy],
                        dtype=cropped.dtype,
                        device=device
                    ).view(1, 2, 1, 1)

                    error = torch.linalg.vector_norm(
                        cropped - target,
                        dim=1
                    )

                    count = error.numel()

                    total_epe += error.sum().item()
                    total_dx += cropped[:, 0].sum().item()
                    total_dy += cropped[:, 1].sum().item()
                    total_pixels += count

            records.append({
                "actual_dx": dx,
                "actual_dy": dy,
                "level": level + 1,
                "predicted_dx": total_dx / total_pixels,
                "predicted_dy": total_dy / total_pixels,
                "EPE": total_epe / total_pixels,
                "zero_flow_EPE": np.hypot(dx, dy)
            })

results_v3 = pd.DataFrame(records)

print("V3 DIRECTIONAL REGISTRATION RESULTS")
print("=" * 75)
print(results_v3.round(4).to_string(index=False))

print("\nMEAN EPE BY LEVEL")
print(
    results_v3.groupby("level")[
        ["EPE", "zero_flow_EPE"]
    ].mean().round(4)
)

save_dir = Path(
    "/content/drive/MyDrive/UA_FloodNet/v3/diagnostics"
)

save_dir.mkdir(parents=True, exist_ok=True)

results_v3.to_csv(
    save_dir / "directional_registration_results.csv",
    index=False
)

print("\nResults saved:", save_dir)

In [ ]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from pathlib import Path
from registration import translate_zero

# Use the existing validation dataset
dataset = dataset_v2

MAX_SHIFT = 4
N_SAMPLES = 10
SHIFTS = [(0, 0), (2, 0), (-2, 0), (0, 2), (0, -2), (2, 2)]

def to_gray(x):
    x = x.detach().cpu().numpy()
    if x.ndim == 3:
        return x.mean(axis=0)
    return x

def gradient_features(image):
    """Gradient magnitude reduces dependence on raw brightness."""
    image = image.astype(np.float32)
    gy, gx = np.gradient(image)
    return np.sqrt(gx**2 + gy**2)

def ncc(a, b):
    a = a.ravel().astype(np.float64)
    b = b.ravel().astype(np.float64)

    a = a - a.mean()
    b = b - b.mean()

    denominator = np.linalg.norm(a) * np.linalg.norm(b)

    if denominator < 1e-10:
        return -1.0

    return float(np.dot(a, b) / denominator)

def estimate_shift(reference, moving, search=4):
    # Fixed interior avoids artificial zero-padding boundaries.
    margin = 2 * search + 4

    reference = gradient_features(reference)
    moving = gradient_features(moving)

    h, w = reference.shape

    fixed = reference[
        margin:h-margin,
        margin:w-margin
    ]

    best_score = -np.inf
    best_shift = (0, 0)

    for dy in range(-search, search + 1):
        for dx in range(-search, search + 1):

            candidate = moving[
                margin+dy:h-margin+dy,
                margin+dx:w-margin+dx
            ]

            score = ncc(fixed, candidate)

            if score > best_score:
                best_score = score
                best_shift = (dx, dy)

    return best_shift, best_score

records = []

for i in range(min(N_SAMPLES, len(dataset))):
    sample = dataset[i]

    sar = to_gray(sample["sar_pre"])
    ms = to_gray(sample["ms_pre"])

    # Establish the unshifted cross-sensor reference offset.
    baseline_cross, _ = estimate_shift(sar, ms, MAX_SHIFT)

    for dx, dy in SHIFTS:

        shifted = translate_zero(
            sample["ms_pre"].unsqueeze(0),
            dx, dy
        )[0]

        shifted_ms = to_gray(shifted)

        same_shift, same_score = estimate_shift(
            ms, shifted_ms, MAX_SHIFT
        )

        cross_shift, cross_score = estimate_shift(
            sar, shifted_ms, MAX_SHIFT
        )

        records.append({
            "sample": i,
            "actual_dx": dx,
            "actual_dy": dy,
            "same_dx": same_shift[0],
            "same_dy": same_shift[1],
            "same_score": same_score,
            "cross_dx": cross_shift[0],
            "cross_dy": cross_shift[1],
            "cross_score": cross_score,
            "cross_baseline_dx": baseline_cross[0],
            "cross_baseline_dy": baseline_cross[1]
        })

results = pd.DataFrame(records)

results["same_error"] = np.hypot(
    results["same_dx"] - results["actual_dx"],
    results["same_dy"] - results["actual_dy"]
)

# Cross-sensor comparison measures response relative
# to the unshifted registration estimate.
results["cross_response_error"] = np.hypot(
    (results["cross_dx"] - results["cross_baseline_dx"])
    - results["actual_dx"],
    (results["cross_dy"] - results["cross_baseline_dy"])
    - results["actual_dy"]
)

print("STEP 36: REGISTRATION DIAGNOSTICS")
print("=" * 65)

print("\nMean same-sensor translation error:")
print(round(results["same_error"].mean(), 4))

print("\nMean cross-sensor translation-response error:")
print(round(results["cross_response_error"].mean(), 4))

print("\nExact same-sensor recovery rate:")
print(round(100 * (results["same_error"] == 0).mean(), 2), "%")

print("\nRESULTS BY TRANSLATION")
summary = results.groupby(
    ["actual_dx", "actual_dy"]
)[["same_error", "cross_response_error"]].mean()

print(summary.round(4).to_string())

print("\nFIRST 12 INDIVIDUAL RESULTS")
print(results.head(12).round(4).to_string(index=False))

save_dir = Path(
    "/content/drive/MyDrive/UA_FloodNet/v3/diagnostics"
)
save_dir.mkdir(parents=True, exist_ok=True)

results.to_csv(
    save_dir / "same_vs_cross_sensor_registration.csv",
    index=False
)

print("\nResults saved:", save_dir)

In [ ]:
import numpy as np
import pandas as pd

# More directions and magnitudes, including 4-pixel shifts.
translations = [
    (0, 0),
    (1, 0), (-1, 0),
    (2, 0), (-2, 0),
    (4, 0), (-4, 0),
    (0, 2), (0, -2),
    (0, 4), (0, -4),
    (2, 2), (-2, -2)
]

records = []

for i in range(len(dataset_v2)):
    sample = dataset_v2[i]

    sar = to_gray(sample["sar_pre"])
    ms = to_gray(sample["ms_pre"])

    baseline, baseline_score = estimate_shift(
        sar, ms, search=4
    )

    for dx, dy in translations:
        shifted = translate_zero(
            sample["ms_pre"].unsqueeze(0),
            dx, dy
        )[0]

        shifted_ms = to_gray(shifted)

        # Use a wider search window to avoid clipping.
        same, same_score = estimate_shift(
            ms, shifted_ms, search=8
        )

        cross, cross_score = estimate_shift(
            sar, shifted_ms, search=8
        )

        same_error = np.hypot(
            same[0] - dx,
            same[1] - dy
        )

        cross_error = np.hypot(
            (cross[0] - baseline[0]) - dx,
            (cross[1] - baseline[1]) - dy
        )

        records.append({
            "sample": i,
            "dx": dx,
            "dy": dy,
            "same_error": same_error,
            "cross_error": cross_error,
            "same_score": same_score,
            "cross_score": cross_score,
            "baseline_dx": baseline[0],
            "baseline_dy": baseline[1]
        })

results = pd.DataFrame(records)

print("STEP 37: FULL VALIDATION DIAGNOSTIC")
print("=" * 60)

print("Total experiments:", len(results))

print(
    "Same-sensor exact recovery:",
    round(100 * (results.same_error == 0).mean(), 2), "%"
)

print(
    "Cross-sensor exact response recovery:",
    round(100 * (results.cross_error == 0).mean(), 2), "%"
)

print("Mean same-sensor error:", round(results.same_error.mean(), 4))
print("Mean cross-sensor error:", round(results.cross_error.mean(), 4))

print("\nERROR BY TRANSLATION")
print(
    results.groupby(["dx", "dy"])[
        ["same_error", "cross_error"]
    ].mean().round(4).to_string()
)

print("\nCROSS-SENSOR CORRELATION SCORE")
print(results.cross_score.describe().round(4))

save_dir = (
    "/content/drive/MyDrive/UA_FloodNet/v4/"
)

import os
os.makedirs(save_dir, exist_ok=True)

results.to_csv(
    save_dir + "full_correspondence_validation.csv",
    index=False
)

print("\nResults saved successfully.")

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from registration import translate_zero

SEARCH_RADIUS = 8

translations = [
    (0, 0),
    (1, 0), (-1, 0),
    (2, 0), (-2, 0),
    (4, 0), (-4, 0),
    (0, 2), (0, -2),
    (0, 4), (0, -4),
    (2, 2), (-2, -2)
]

records = []

for i in range(len(dataset_v2)):
    sample = dataset_v2[i]

    sar = to_gray(sample["sar_pre"])
    ms = to_gray(sample["ms_pre"])

    # Critical correction: same search radius for both estimates.
    baseline, baseline_score = estimate_shift(
        sar, ms, search=SEARCH_RADIUS
    )

    for dx, dy in translations:

        shifted = translate_zero(
            sample["ms_pre"].unsqueeze(0),
            dx, dy
        )[0]

        shifted_ms = to_gray(shifted)

        estimate, score = estimate_shift(
            sar, shifted_ms, search=SEARCH_RADIUS
        )

        predicted_dx = estimate[0] - baseline[0]
        predicted_dy = estimate[1] - baseline[1]

        error = np.hypot(
            predicted_dx - dx,
            predicted_dy - dy
        )

        # Would the ideal translated baseline offset
        # lie outside the permitted search window?
        expected_x = baseline[0] + dx
        expected_y = baseline[1] + dy

        in_range = (
            abs(expected_x) <= SEARCH_RADIUS
            and abs(expected_y) <= SEARCH_RADIUS
        )

        boundary_hit = (
            abs(estimate[0]) == SEARCH_RADIUS
            or abs(estimate[1]) == SEARCH_RADIUS
        )

        records.append({
            "sample": i,
            "actual_dx": dx,
            "actual_dy": dy,
            "baseline_dx": baseline[0],
            "baseline_dy": baseline[1],
            "estimated_dx": estimate[0],
            "estimated_dy": estimate[1],
            "response_dx": predicted_dx,
            "response_dy": predicted_dy,
            "error": error,
            "expected_in_range": in_range,
            "boundary_hit": boundary_hit,
            "baseline_score": baseline_score,
            "shifted_score": score
        })

audit = pd.DataFrame(records)

def report_subset(name, frame):
    print(f"\n{name}")
    print("-" * 60)

    if len(frame) == 0:
        print("No experiments in this subset.")
        return

    print("Number of experiments:", len(frame))
    print(
        "Exact recovery (%):",
        round(100 * (frame["error"] == 0).mean(), 2)
    )
    print(
        "Mean error (pixels):",
        round(frame["error"].mean(), 4)
    )
    print(
        "Median error (pixels):",
        round(frame["error"].median(), 4)
    )

print("STEP 38: CORRECTED CROSS-SENSOR REGISTRATION AUDIT")
print("=" * 65)

report_subset("ALL EXPERIMENTS", audit)

report_subset(
    "EXPECTED DISPLACEMENT INSIDE SEARCH WINDOW",
    audit[audit["expected_in_range"]]
)

report_subset(
    "INSIDE SEARCH WINDOW WITHOUT BOUNDARY HITS",
    audit[
        audit["expected_in_range"]
        & ~audit["boundary_hit"]
    ]
)

report_subset(
    "ZERO-SHIFT CONSISTENCY TEST",
    audit[
        (audit["actual_dx"] == 0)
        & (audit["actual_dy"] == 0)
    ]
)

print("\nMEAN ERROR BY TRANSLATION")
print(
    audit.groupby(["actual_dx", "actual_dy"])["error"]
    .agg(["mean", "median", "max"])
    .round(4)
    .to_string()
)

print("\nSEARCH WINDOW DIAGNOSTICS")
print(
    "Expected offsets outside search:",
    int((~audit["expected_in_range"]).sum())
)
print(
    "Estimated offsets at search boundary:",
    int(audit["boundary_hit"].sum())
)

output_dir = Path(
    "/content/drive/MyDrive/UA_FloodNet/v4/diagnostics"
)
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "step38_corrected_registration.csv"
audit.to_csv(output_path, index=False)

print("\nSaved to:", output_path)

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

# One original SAR–MS estimate per validation sample
baseline = (
    audit.groupby("sample", as_index=False)
    .agg(
        baseline_dx=("baseline_dx", "first"),
        baseline_dy=("baseline_dy", "first"),
        baseline_score=("baseline_score", "first")
    )
)

baseline["offset_magnitude"] = np.hypot(
    baseline["baseline_dx"],
    baseline["baseline_dy"]
)

baseline["at_search_boundary"] = (
    (baseline["baseline_dx"].abs() == 8)
    | (baseline["baseline_dy"].abs() == 8)
)

print("STEP 39: BASELINE CROSS-SENSOR OFFSET AUDIT")
print("=" * 65)

print("Validation images:", len(baseline))

print("\nBASELINE OFFSET STATISTICS")
print(
    baseline[
        ["baseline_dx", "baseline_dy",
         "offset_magnitude", "baseline_score"]
    ].describe().round(4)
)

print("\nOFFSET MAGNITUDE DISTRIBUTION")
print(
    pd.cut(
        baseline["offset_magnitude"],
        bins=[-0.01, 1, 2, 4, 8, 12],
        labels=["0–1", "1–2", "2–4", "4–8", "8–12"],
        include_lowest=True
    ).value_counts().sort_index()
)

print("\nBASELINE ESTIMATES AT SEARCH BOUNDARY")
print(int(baseline["at_search_boundary"].sum()))

print("\nMOST QUESTIONABLE CORRESPONDENCES")
print(
    baseline.sort_values("baseline_score")
    .head(15)
    .round(4)
    .to_string(index=False)
)

print("\nLARGEST ESTIMATED OFFSETS")
print(
    baseline.sort_values(
        "offset_magnitude", ascending=False
    )
    .head(15)
    .round(4)
    .to_string(index=False)
)

save_dir = Path(
    "/content/drive/MyDrive/UA_FloodNet/v4/diagnostics"
)
save_dir.mkdir(parents=True, exist_ok=True)

baseline.to_csv(
    save_dir / "step39_baseline_offsets.csv",
    index=False
)

print("\nBaseline audit saved successfully.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch

# Requires dataset_v2, baseline, to_gray,
# translate_zero and gradient_features from earlier steps.

sample_ids = [34, 32, 30, 79, 99, 64]

fig, axes = plt.subplots(
    len(sample_ids), 3,
    figsize=(13, 3.5 * len(sample_ids))
)

for row, sample_id in enumerate(sample_ids):
    sample = dataset_v2[sample_id]

    sar = to_gray(sample["sar_pre"])
    ms = to_gray(sample["ms_pre"])

    record = baseline.loc[
        baseline["sample"] == sample_id
    ].iloc[0]

    dx = int(record["baseline_dx"])
    dy = int(record["baseline_dy"])
    score = float(record["baseline_score"])

    # Estimated offsets describe where the moving image
    # is located relative to the reference.
    # Apply the inverse translation for visual alignment.
    corrected = translate_zero(
        sample["ms_pre"].unsqueeze(0),
        -dx, -dy
    )[0]

    corrected_ms = to_gray(corrected)

    def normalized_edges(image):
        edges = gradient_features(image)
        return edges / (np.percentile(edges, 99) + 1e-6)

    sar_edges = normalized_edges(sar)
    ms_edges = normalized_edges(ms)
    corrected_edges = normalized_edges(corrected_ms)

    def overlay(a, b):
        return np.stack([
            np.clip(a, 0, 1),
            np.clip(b, 0, 1),
            np.zeros_like(a)
        ], axis=-1)

    axes[row, 0].imshow(sar, cmap="gray")
    axes[row, 0].set_title(f"Sample {sample_id}: SAR")

    axes[row, 1].imshow(
        overlay(sar_edges, ms_edges)
    )
    axes[row, 1].set_title("Before alignment")

    axes[row, 2].imshow(
        overlay(sar_edges, corrected_edges)
    )
    axes[row, 2].set_title(
        f"After ({-dx:+d}, {-dy:+d}) px\n"
        f"Original NCC = {score:.3f}"
    )

    for ax in axes[row]:
        ax.axis("off")

plt.tight_layout()
plt.savefig(
    "/content/drive/MyDrive/UA_FloodNet/v4/"
    "diagnostics/step40_alignment_inspection.png",
    dpi=250,
    bbox_inches="tight"
)
plt.show()

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

# Uses dataset_v2, baseline, to_gray,
# gradient_features and ncc from earlier steps.

MARGIN = 20
records = []

for _, rec in baseline.iterrows():
    i = int(rec["sample"])
    dx = int(rec["baseline_dx"])
    dy = int(rec["baseline_dy"])

    sample = dataset_v2[i]
    sar = gradient_features(to_gray(sample["sar_pre"]))
    ms = gradient_features(to_gray(sample["ms_pre"]))

    h, w = sar.shape

    # Same fixed reference region for both comparisons.
    ref = sar[MARGIN:h-MARGIN, MARGIN:w-MARGIN]

    # No image translation/padding: sample directly
    # at the estimated corresponding coordinates.
    before = ms[MARGIN:h-MARGIN, MARGIN:w-MARGIN]

    after = ms[
        MARGIN+dy:h-MARGIN+dy,
        MARGIN+dx:w-MARGIN+dx
    ]

    score_before = ncc(ref, before)
    score_after = ncc(ref, after)

    records.append({
        "sample": i,
        "dx": dx,
        "dy": dy,
        "offset_px": np.hypot(dx, dy),
        "ncc_before": score_before,
        "ncc_after": score_after,
        "ncc_gain": score_after - score_before,
        "boundary": abs(dx) == 8 or abs(dy) == 8
    })

quality = pd.DataFrame(records)

print("STEP 41: ALIGNMENT QUALITY AUDIT")
print("=" * 65)

print("\nMEAN NCC BEFORE / AFTER")
print(quality[["ncc_before", "ncc_after", "ncc_gain"]].mean().round(4))

print("\nSUSPICIOUS SAMPLES")
print(
    quality[quality["sample"].isin([34, 32, 30, 79, 99, 64])]
    .round(4).to_string(index=False)
)

print("\nNUMBER WITH POSITIVE NCC GAIN")
print(int((quality.ncc_gain > 1e-6).sum()), "/", len(quality))

print("\nLARGE OFFSETS (>4 PIXELS)")
print(
    quality[quality.offset_px > 4]
    .round(4).to_string(index=False)
)

output_dir = Path(
    "/content/drive/MyDrive/UA_FloodNet/v4/diagnostics"
)
output_dir.mkdir(parents=True, exist_ok=True)

quality.to_csv(
    output_dir / "step41_alignment_quality.csv",
    index=False
)

print("\nResults saved.")

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

v4_gate = quality.copy()

MAX_CORRECTION = 2.0
MIN_NCC = 0.20
MIN_GAIN = 0.02

v4_gate["accept"] = (
    (v4_gate["offset_px"] > 0)
    & (v4_gate["offset_px"] <= MAX_CORRECTION)
    & (v4_gate["ncc_after"] >= MIN_NCC)
    & (v4_gate["ncc_gain"] >= MIN_GAIN)
    & (~v4_gate["boundary"])
)

v4_gate["reason"] = np.select(
    [
        v4_gate["offset_px"] == 0,
        v4_gate["boundary"],
        v4_gate["offset_px"] > MAX_CORRECTION,
        v4_gate["ncc_after"] < MIN_NCC,
        v4_gate["ncc_gain"] < MIN_GAIN,
    ],
    [
        "Already zero offset",
        "Search boundary",
        "Exceeds correction limit",
        "Weak correspondence",
        "Insufficient NCC gain",
    ],
    default="Accepted"
)

print("STEP 42: V4 CONFIDENCE-GATED REGISTRATION")
print("=" * 65)

print("Validation images:", len(v4_gate))
print("Corrections accepted:", int(v4_gate.accept.sum()))
print("Original images retained:", int((~v4_gate.accept).sum()))

print("\nDECISION COUNTS")
print(v4_gate["reason"].value_counts().to_string())

print("\nACCEPTED CORRECTIONS")
print(
    v4_gate.loc[
        v4_gate.accept,
        ["sample", "dx", "dy", "offset_px",
         "ncc_before", "ncc_after", "ncc_gain"]
    ].round(4).to_string(index=False)
)

print("\nPREVIOUSLY SUSPICIOUS CASES")
print(
    v4_gate[
        v4_gate["sample"].isin([30, 32, 34, 64, 79, 99])
    ][
        ["sample", "dx", "dy", "ncc_after",
         "ncc_gain", "accept", "reason"]
    ].round(4).to_string(index=False)
)

output_dir = Path(
    "/content/drive/MyDrive/UA_FloodNet/v4/diagnostics"
)
output_dir.mkdir(parents=True, exist_ok=True)

v4_gate.to_csv(
    output_dir / "step42_registration_gate.csv",
    index=False
)

print("\nStep 42 completed. Decisions saved.")

In [ ]:
import sys
import numpy as np
import pandas as pd
import torch
from pathlib import Path

sys.path.insert(0, "/content/UA_FloodNet_V2")

from ablation_model import UAFloodNet
from registration import translate_zero

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

checkpoint_path = (
    "/content/drive/MyDrive/UA_FloodNet/"
    "ablations/full_seed42.pt"
)

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=True
)

print("Checkpoint keys:", list(checkpoint.keys()))

model = UAFloodNet(
    width=24,
    use_alignment=True,
    use_reliability=True
).to(device)

model.load_state_dict(checkpoint["model"])
model.eval()

# Use the same dataset object as Steps 36–42
# to preserve exact sample indexing.
assert len(dataset_v2) == len(v4_gate) == 125

decisions = v4_gate.set_index("sample")
assert set(decisions.index) == set(range(len(dataset_v2)))

def get_logits(output):
    if isinstance(output, torch.Tensor):
        return output

    if isinstance(output, dict):
        for key in ["logits", "seg_logits", "mask_logits"]:
            if key in output:
                return output[key]

    raise ValueError(
        "Unexpected model output. Inspect output keys "
        "before continuing."
    )

def compute_counts(pred, truth, valid):
    pred = pred.bool() & valid
    truth = truth.bool() & valid

    tp = (pred & truth).sum().item()
    fp = (pred & ~truth & valid).sum().item()
    fn = (~pred & truth & valid).sum().item()

    return tp, fp, fn

def calculate_scores(tp, fp, fn):
    iou = tp / (tp + fp + fn + 1e-8)
    f1 = 2 * tp / (2 * tp + fp + fn + 1e-8)
    return iou, f1

records = []

with torch.inference_mode():

    for i in range(len(dataset_v2)):

        sample = dataset_v2[i]
        decision = decisions.loc[i]

        original = {
            k: sample[k].unsqueeze(0).to(device)
            for k in [
                "sar_pre", "sar_post",
                "ms_pre", "ms_post"
            ]
        }

        truth = (
            sample["mask"].unsqueeze(0).to(device) > 0.5
        )

        corrected = {
            k: v.clone() for k, v in original.items()
        }

        h, w = truth.shape[-2:]
        valid = torch.ones_like(truth, dtype=torch.bool)

        accepted = bool(decision["accept"])
        dx = int(decision["dx"]) if accepted else 0
        dy = int(decision["dy"]) if accepted else 0

        if accepted:
            corrected["ms_pre"] = translate_zero(
                corrected["ms_pre"], -dx, -dy
            )
            corrected["ms_post"] = translate_zero(
                corrected["ms_post"], -dx, -dy
            )

            # Pixels newly exposed by translation are invalid.
            valid_float = torch.ones(
                (1, 1, h, w), device=device
            )
            valid = translate_zero(
                valid_float, -dx, -dy
            ) > 0.5

        original_output = model(**original)
        corrected_output = model(**corrected)

        original_pred = (
            torch.sigmoid(get_logits(original_output)) >= 0.5
        )
        corrected_pred = (
            torch.sigmoid(get_logits(corrected_output)) >= 0.5
        )

        if original_pred.shape != truth.shape:
            raise ValueError(
                f"Prediction/mask shapes differ: "
                f"{original_pred.shape} vs {truth.shape}"
            )

        # Matched valid region for a fair comparison.
        tp0, fp0, fn0 = compute_counts(
            original_pred, truth, valid
        )
        tp1, fp1, fn1 = compute_counts(
            corrected_pred, truth, valid
        )

        iou0, f10 = calculate_scores(tp0, fp0, fn0)
        iou1, f11 = calculate_scores(tp1, fp1, fn1)

        records.append({
            "sample": i,
            "accepted": accepted,
            "dx": dx,
            "dy": dy,
            "valid_fraction": valid.float().mean().item(),
            "original_iou": iou0,
            "registered_iou": iou1,
            "delta_iou": iou1 - iou0,
            "original_f1": f10,
            "registered_f1": f11,
            "delta_f1": f11 - f10,
            "tp_original": tp0,
            "fp_original": fp0,
            "fn_original": fn0,
            "tp_registered": tp1,
            "fp_registered": fp1,
            "fn_registered": fn1
        })

results = pd.DataFrame(records)

def pooled_metrics(df, suffix):
    tp = df[f"tp_{suffix}"].sum()
    fp = df[f"fp_{suffix}"].sum()
    fn = df[f"fn_{suffix}"].sum()
    return calculate_scores(tp, fp, fn)

print("\nSTEP 43: REGISTRATION EFFECT ON FLOOD DETECTION")
print("=" * 65)

for name, subset in [
    ("ALL VALIDATION IMAGES", results),
    ("ACCEPTED CORRECTIONS ONLY", results[results.accepted]),
    ("UNCHANGED IMAGES", results[~results.accepted])
]:
    print("\n", name)
    print("-" * 55)

    orig_iou, orig_f1 = pooled_metrics(subset, "original")
    reg_iou, reg_f1 = pooled_metrics(subset, "registered")

    print("Images:", len(subset))
    print("Original pooled IoU:", round(orig_iou, 4))
    print("Registered pooled IoU:", round(reg_iou, 4))
    print("Delta pooled IoU:", round(reg_iou - orig_iou, 4))
    print("Original pooled F1:", round(orig_f1, 4))
    print("Registered pooled F1:", round(reg_f1, 4))
    print("Delta pooled F1:", round(reg_f1 - orig_f1, 4))

print("\nACCEPTED IMAGES: INDIVIDUAL RESULTS")
print(
    results[results.accepted][[
        "sample", "dx", "dy",
        "original_iou", "registered_iou", "delta_iou"
    ]].round(4).to_string(index=False)
)

print("\nACCEPTED IMAGES IMPROVED / WORSENED / UNCHANGED")
accepted_results = results[results.accepted]
print(
    "Improved:",
    int((accepted_results.delta_iou > 1e-6).sum())
)
print(
    "Worsened:",
    int((accepted_results.delta_iou < -1e-6).sum())
)
print(
    "Unchanged:",
    int((accepted_results.delta_iou.abs() <= 1e-6).sum())
)

save_dir = Path(
    "/content/drive/MyDrive/UA_FloodNet/v4/diagnostics"
)
save_dir.mkdir(parents=True, exist_ok=True)

results.to_csv(
    save_dir / "step43_registration_vs_original.csv",
    index=False
)

print("\nResults saved:", save_dir)

In [ ]:
import sys
import numpy as np
import pandas as pd
import torch
from pathlib import Path

sys.path.insert(0, "/content/UA_FloodNet_V2")

from ablation_model import UAFloodNet
from registration import translate_zero

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------------------------------------------------------
# 1. Load existing flood model
# ---------------------------------------------------------
ckpt = torch.load(
    "/content/drive/MyDrive/UA_FloodNet/ablations/full_seed42.pt",
    map_location="cpu",
    weights_only=True
)

model = UAFloodNet(
    width=ckpt["width"],
    use_alignment=True,
    use_reliability=True
).to(device)

model.load_state_dict(ckpt["model"])
model.eval()

print("Loaded checkpoint epoch:", ckpt["epoch"])
print("Evaluation device:", device)

# ---------------------------------------------------------
# 2. Experiment settings
# ---------------------------------------------------------
SHIFTS = [
    (1, 0), (-1, 0),
    (2, 0), (-2, 0),
    (4, 0), (-4, 0),
    (0, 2), (0, -2)
]

SEARCH_RADIUS = 8
MARGIN = 20
THRESHOLD = 0.50

assert len(dataset_v2) == 125

def model_prob(inputs):
    output = model(**inputs)

    if torch.is_tensor(output):
        logits = output
    elif isinstance(output, dict):
        logits = None
        for key in ("logits", "seg_logits", "mask_logits"):
            if key in output:
                logits = output[key]
                break
        if logits is None:
            raise ValueError(f"Unknown output keys: {list(output)}")
    else:
        raise TypeError("Unexpected model output type")

    return torch.sigmoid(logits)

def apply_ms_shift(inputs, dx, dy):
    modified = dict(inputs)
    modified["ms_pre"] = translate_zero(inputs["ms_pre"], dx, dy)
    modified["ms_post"] = translate_zero(inputs["ms_post"], dx, dy)
    return modified

def confusion_counts(prob, truth, valid):
    pred = prob >= THRESHOLD
    target = truth > 0.5

    tp = ((pred & target) & valid).sum().item()
    fp = ((pred & ~target) & valid).sum().item()
    fn = ((~pred & target) & valid).sum().item()

    return int(tp), int(fp), int(fn)

def scores(tp, fp, fn):
    denominator = tp + fp + fn
    iou = tp / denominator if denominator else np.nan
    denominator_f1 = 2 * tp + fp + fn
    f1 = 2 * tp / denominator_f1 if denominator_f1 else np.nan
    return iou, f1

# Exclude borders in all conditions, including clean.
# With a maximum synthetic shift of 4 and search of 8,
# a 20-pixel margin avoids artificial translated edges.
records = []

# ---------------------------------------------------------
# 3. Run all comparisons
# ---------------------------------------------------------
with torch.inference_mode():

    for i in range(len(dataset_v2)):
        sample = dataset_v2[i]

        clean = {
            k: sample[k].unsqueeze(0).to(device)
            for k in ("sar_pre", "sar_post", "ms_pre", "ms_post")
        }

        truth = sample["mask"].unsqueeze(0).to(device)
        h, w = truth.shape[-2:]

        valid = torch.zeros_like(truth, dtype=torch.bool)
        valid[:, :, MARGIN:h-MARGIN, MARGIN:w-MARGIN] = True

        original_prob = model_prob(clean)

        # Use the same estimated original offset to evaluate
        # recovery of additional synthetic displacement.
        sar_np = to_gray(sample["sar_pre"])
        ms_np = to_gray(sample["ms_pre"])

        baseline_offset, baseline_score = estimate_shift(
            sar_np, ms_np, search=SEARCH_RADIUS
        )

        for dx, dy in SHIFTS:

            shifted = apply_ms_shift(clean, dx, dy)

            # Undo the known imposed displacement.
            oracle = apply_ms_shift(shifted, -dx, -dy)

            shifted_ms_np = to_gray(shifted["ms_pre"][0])

            estimated_offset, estimated_score = estimate_shift(
                sar_np,
                shifted_ms_np,
                search=SEARCH_RADIUS
            )

            # Practical estimate: use SAR-MS offset directly.
            # This assumes original image alignment is accurate.
            practical = apply_ms_shift(
                shifted,
                -estimated_offset[0],
                -estimated_offset[1]
            )

            # Diagnostic only: original baseline is available
            # because this is a controlled experiment.
            # It cannot be assumed known in real deployment.
            relative_dx = estimated_offset[0] - baseline_offset[0]
            relative_dy = estimated_offset[1] - baseline_offset[1]

            relative = apply_ms_shift(
                shifted, -relative_dx, -relative_dy
            )

            predictions = {
                "clean": original_prob,
                "shifted": model_prob(shifted),
                "oracle": model_prob(oracle),
                "practical": model_prob(practical),
                "relative_diagnostic": model_prob(relative)
            }

            for condition, probability in predictions.items():
                tp, fp, fn = confusion_counts(
                    probability, truth, valid
                )

                records.append({
                    "sample": i,
                    "shift_dx": dx,
                    "shift_dy": dy,
                    "condition": condition,
                    "tp": tp,
                    "fp": fp,
                    "fn": fn,
                    "baseline_dx": baseline_offset[0],
                    "baseline_dy": baseline_offset[1],
                    "estimated_dx": estimated_offset[0],
                    "estimated_dy": estimated_offset[1],
                    "relative_dx": relative_dx,
                    "relative_dy": relative_dy,
                    "baseline_ncc": baseline_score,
                    "shifted_ncc": estimated_score,
                    "relative_EPE": float(np.hypot(
                        relative_dx - dx, relative_dy - dy
                    ))
                })

        if (i + 1) % 25 == 0:
            print(f"Processed {i+1}/{len(dataset_v2)} images")

# ---------------------------------------------------------
# 4. Summarise pooled IoU and F1
# ---------------------------------------------------------
results44 = pd.DataFrame(records)

summary_rows = []

for (dx, dy, condition), group in results44.groupby(
    ["shift_dx", "shift_dy", "condition"]
):
    tp = group.tp.sum()
    fp = group.fp.sum()
    fn = group.fn.sum()

    iou, f1 = scores(tp, fp, fn)

    summary_rows.append({
        "shift_dx": dx,
        "shift_dy": dy,
        "condition": condition,
        "pooled_iou": iou,
        "pooled_f1": f1
    })

summary44 = pd.DataFrame(summary_rows)

print("\nSTEP 44: CONTROLLED MISREGISTRATION ROBUSTNESS")
print("=" * 75)

print("\nMEAN POOLED IoU ACROSS SHIFTS")
print(
    summary44.groupby("condition")[
        ["pooled_iou", "pooled_f1"]
    ].mean().round(4).to_string()
)

print("\nIoU BY KNOWN TRANSLATION")
print(
    summary44.pivot_table(
        index=["shift_dx", "shift_dy"],
        columns="condition",
        values="pooled_iou"
    ).round(4).to_string()
)

# ---------------------------------------------------------
# 5. Compare correction recovery
# ---------------------------------------------------------
pivot = summary44.pivot_table(
    index=["shift_dx", "shift_dy"],
    columns="condition",
    values="pooled_iou"
)

for condition in ("oracle", "practical", "relative_diagnostic"):
    pivot[condition + "_gain"] = (
        pivot[condition] - pivot["shifted"]
    )

print("\nIoU RECOVERY RELATIVE TO SHIFTED INPUTS")
print(
    pivot[[
        "oracle_gain",
        "practical_gain",
        "relative_diagnostic_gain"
    ]].round(4).to_string()
)

print("\nRELATIVE SHIFT ESTIMATION ACCURACY")
diagnostic = results44[
    results44.condition == "clean"
]

print(
    "Mean relative EPE:",
    round(diagnostic.relative_EPE.mean(), 4)
)

print(
    "Exact relative recovery (%):",
    round(
        100 * (diagnostic.relative_EPE == 0).mean(),
        2
    )
)

# ---------------------------------------------------------
# 6. Save all results
# ---------------------------------------------------------
output_dir = Path(
    "/content/drive/MyDrive/UA_FloodNet/v4/diagnostics"
)
output_dir.mkdir(parents=True, exist_ok=True)

results44.to_csv(
    output_dir / "step44_detailed_results.csv",
    index=False
)

summary44.to_csv(
    output_dir / "step44_summary.csv",
    index=False
)

pivot.to_csv(
    output_dir / "step44_recovery_comparison.csv"
)

print("\nStep 44 completed. All CSV files saved.")

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

path = Path(
    "/content/drive/MyDrive/UA_FloodNet/v4/"
    "diagnostics/step44_detailed_results.csv"
)

df = pd.read_csv(path)

rows = []

for (sample, dx, dy), group in df.groupby(
    ["sample", "shift_dx", "shift_dy"]
):
    metrics = {}

    for condition, rec in group.set_index("condition").iterrows():
        tp, fp, fn = rec[["tp", "fp", "fn"]]
        denom = tp + fp + fn

        metrics[condition] = (
            tp / denom if denom > 0 else np.nan
        )

    record = group.iloc[0]

    rows.append({
        "sample": sample,
        "dx": dx,
        "dy": dy,
        "baseline_dx": record["baseline_dx"],
        "baseline_dy": record["baseline_dy"],
        "estimated_dx": record["estimated_dx"],
        "estimated_dy": record["estimated_dy"],
        "baseline_ncc": record["baseline_ncc"],
        "shifted_ncc": record["shifted_ncc"],
        "relative_EPE": record["relative_EPE"],
        "clean_iou": metrics["clean"],
        "shifted_iou": metrics["shifted"],
        "practical_iou": metrics["practical"],
        "relative_iou": metrics["relative_diagnostic"],
        "oracle_iou": metrics["oracle"]
    })

diagnostics = pd.DataFrame(rows)

diagnostics["practical_gain"] = (
    diagnostics["practical_iou"]
    - diagnostics["shifted_iou"]
)

diagnostics["practical_vs_clean"] = (
    diagnostics["practical_iou"]
    - diagnostics["clean_iou"]
)

diagnostics["relative_gain"] = (
    diagnostics["relative_iou"]
    - diagnostics["shifted_iou"]
)

diagnostics["practical_vs_relative"] = (
    diagnostics["practical_iou"]
    - diagnostics["relative_iou"]
)

print("STEP 45: REGISTRATION FAILURE ANALYSIS")
print("=" * 65)

print("\nNUMBER OF SAMPLE-SHIFT EXPERIMENTS")
print(len(diagnostics))

print("\nPRACTICAL CORRECTION")
print(
    "Improved:",
    int((diagnostics.practical_gain > 1e-6).sum())
)
print(
    "Worsened:",
    int((diagnostics.practical_gain < -1e-6).sum())
)
print(
    "Unchanged:",
    int((diagnostics.practical_gain.abs() <= 1e-6).sum())
)

print("\nMEAN PER-SAMPLE IoU DIFFERENCES")
print(
    diagnostics[
        ["practical_gain", "relative_gain",
         "practical_vs_clean", "practical_vs_relative"]
    ].mean().round(5)
)

print("\nWORST PRACTICAL CORRECTION CASES")
print(
    diagnostics.sort_values("practical_gain")
    .head(20)
    .round(4)
    .to_string(index=False)
)

print("\nCASES WITH LARGE RELATIVE REGISTRATION ERROR")
print(
    diagnostics[
        diagnostics.relative_EPE > 1
    ].sort_values("relative_EPE", ascending=False)
    .head(20)
    .round(4)
    .to_string(index=False)
)

print("\nSUMMARY BY TRANSLATION")
print(
    diagnostics.groupby(["dx", "dy"])[
        ["practical_gain", "relative_gain", "relative_EPE"]
    ].mean().round(4).to_string()
)

out = Path(
    "/content/drive/MyDrive/UA_FloodNet/v4/"
    "diagnostics/step45_failure_analysis.csv"
)

diagnostics.to_csv(out, index=False)
print("\nResults saved:", out)

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

folder = Path(
    "/content/drive/MyDrive/UA_FloodNet/v4/diagnostics"
)

df = pd.read_csv(
    folder / "step44_detailed_results.csv"
)

# ---------------------------------------------------------
# 1. Registration confidence policy
# ---------------------------------------------------------
MIN_NCC = 0.20
MAX_OFFSET = 4.0
MAX_NCC_DROP = 0.02

keys = ["sample", "shift_dx", "shift_dy"]

# Each experiment has multiple prediction conditions.
# Keep only one record per sample-shift pair.
decisions = (
    df[df["condition"] == "shifted"]
    .copy()
)

assert len(decisions) == 1000
assert not decisions.duplicated(keys).any()

decisions["estimated_magnitude"] = np.hypot(
    decisions["estimated_dx"],
    decisions["estimated_dy"]
)

decisions["ncc_drop"] = (
    decisions["baseline_ncc"]
    - decisions["shifted_ncc"]
)

decisions["safe_correction"] = (
    (decisions["estimated_magnitude"] > 0)
    & (decisions["estimated_magnitude"] <= MAX_OFFSET)
    & (decisions["shifted_ncc"] >= MIN_NCC)
    & (decisions["ncc_drop"] <= MAX_NCC_DROP)
    & (decisions["estimated_dx"].abs() < 8)
    & (decisions["estimated_dy"].abs() < 8)
)

decisions = decisions[
    keys + [
        "safe_correction",
        "estimated_magnitude",
        "baseline_ncc",
        "shifted_ncc"
    ]
].copy()

# ---------------------------------------------------------
# 2. Prepare shifted and registered predictions
# ---------------------------------------------------------
shifted = (
    df[df["condition"] == "shifted"]
    .copy()
    .sort_values(keys)
    .reset_index(drop=True)
)

practical = (
    df[df["condition"] == "practical"]
    .copy()
    .sort_values(keys)
    .reset_index(drop=True)
)

decisions = (
    decisions
    .sort_values(keys)
    .reset_index(drop=True)
)

assert shifted[keys].equals(practical[keys])
assert shifted[keys].equals(decisions[keys])

# ---------------------------------------------------------
# 3. Construct confidence-gated predictions
# ---------------------------------------------------------
gated = shifted.copy()
gated["condition"] = "gated"

gated["safe_correction"] = (
    decisions["safe_correction"].to_numpy()
)

for col in ["tp", "fp", "fn"]:
    gated[col] = np.where(
        gated["safe_correction"].to_numpy(),
        practical[col].to_numpy(),
        shifted[col].to_numpy()
    )

# ---------------------------------------------------------
# 4. Combine experimental conditions
# ---------------------------------------------------------
combined = pd.concat(
    [
        df[df["condition"].isin([
            "clean",
            "shifted",
            "practical",
            "oracle"
        ])],
        gated
    ],
    ignore_index=True
)

def pooled_metrics(group):
    tp = group["tp"].sum()
    fp = group["fp"].sum()
    fn = group["fn"].sum()

    iou = tp / (tp + fp + fn + 1e-8)
    f1 = 2 * tp / (2 * tp + fp + fn + 1e-8)

    return pd.Series({
        "IoU": iou,
        "F1": f1
    })

# ---------------------------------------------------------
# 5. Overall pooled evaluation
# ---------------------------------------------------------
summary = (
    combined.groupby("condition")
    .apply(
        pooled_metrics,
        include_groups=False
    )
    .sort_values("IoU", ascending=False)
)

print("STEP 46: CONSERVATIVE REGISTRATION POLICY")
print("=" * 65)

print("\nREGISTRATION DECISIONS")
print("Total experiments:", len(decisions))
print(
    "Accepted:",
    int(decisions["safe_correction"].sum())
)
print(
    "Rejected:",
    int((~decisions["safe_correction"]).sum())
)

print("\nPOOLED PERFORMANCE")
print(summary.round(5).to_string())

# ---------------------------------------------------------
# 6. Performance by synthetic displacement
# ---------------------------------------------------------
by_shift = (
    combined.groupby([
        "shift_dx",
        "shift_dy",
        "condition"
    ])
    .apply(
        pooled_metrics,
        include_groups=False
    )
    .reset_index()
)

print("\nPER-SHIFT POOLED IoU")
print(
    by_shift.pivot_table(
        index=["shift_dx", "shift_dy"],
        columns="condition",
        values="IoU"
    ).round(5).to_string()
)

# ---------------------------------------------------------
# 7. Verification checks
# ---------------------------------------------------------
print("\nVERIFICATION")

print(
    "Gated rows:",
    len(gated)
)

print(
    "Accepted decisions:",
    gated["safe_correction"].sum()
)

assert len(gated) == 1000

assert len(combined) == 5000

# ---------------------------------------------------------
# 8. Save results
# ---------------------------------------------------------
summary.to_csv(
    folder / "step46_policy_summary.csv"
)

by_shift.to_csv(
    folder / "step46_policy_by_shift.csv",
    index=False
)

decisions.to_csv(
    folder / "step46_policy_decisions.csv",
    index=False
)

print("\nStep 46 completed successfully.")
print("Results saved to:", folder)

In [ ]:
%cd /content/UA_FloodNet/ablations_code

!python train_ablation.py --help

In [ ]:
import os

for root, dirs, files in os.walk("/content"):
    if "train_ablation.py" in files:
        print(os.path.join(root, "train_ablation.py"))

In [ ]:
from google.colab import drive
from pathlib import Path
import zipfile
import os

# 1. Reconnect Google Drive
drive.mount("/content/drive")

# 2. Check uploaded ZIP files
archives = [
    Path("/content/UA_FloodNet_Ablations.zip"),
    Path("/content/UA_FloodNet_V2.zip")
]

print("\nCHECKING ARCHIVES")
for archive in archives:
    print(archive.name, "Exists:", archive.exists())

# 3. Inspect archive contents before extraction
for archive in archives:
    if archive.exists():
        print(f"\nCONTENTS OF {archive.name}")
        with zipfile.ZipFile(archive) as z:
            for filename in z.namelist()[:25]:
                print(filename)

# 4. Restore code to separate directories
destinations = [
    Path("/content/UA_FloodNet/ablations_code"),
    Path("/content/UA_FloodNet_V2")
]

for archive, destination in zip(archives, destinations):
    if archive.exists():
        destination.mkdir(parents=True, exist_ok=True)

        with zipfile.ZipFile(archive) as z:
            # Reject unsafe archive paths.
            root = destination.resolve()
            for item in z.infolist():
                target = (destination / item.filename).resolve()
                if not target.is_relative_to(root):
                    raise ValueError(
                        f"Unsafe ZIP entry: {item.filename}"
                    )
            z.extractall(destination)

        print(f"\nExtracted: {archive.name}")
        print(f"Location: {destination}")

# 5. Locate restored Python scripts
print("\nRESTORED TRAINING SCRIPTS")
print("=" * 65)

for script in Path("/content").rglob("train_ablation.py"):
    print("FOUND:", script)

for script in Path("/content").rglob("train_v2.py"):
    print("FOUND:", script)

# 6. Verify saved checkpoints
print("\nGOOGLE DRIVE CHECKPOINTS")
print("=" * 65)

checkpoint_paths = [
    "/content/drive/MyDrive/UA_FloodNet/ablations/full_seed42.pt",
    "/content/drive/MyDrive/UA_FloodNet/v3/registration_20epochs.pt"
]

for p in checkpoint_paths:
    print(Path(p).name, "Exists:", Path(p).exists())

print("\nEnvironment restoration check completed.")

In [ ]:
from pathlib import Path

print("STEP 49: DATASET AND TRAINING SCRIPT VERIFICATION")
print("=" * 70)

# 1. Check OMBRIA datasets
data_root = Path("/content/UA_FloodNet/dataset")

for split in ["train", "val", "test"]:
    folder = data_root / split

    if folder.exists():
        files = list(folder.glob("*.npz"))
        print(f"{split.upper()}: {len(files)} NPZ files")
    else:
        print(f"{split.upper()}: NOT FOUND")

# 2. Search for dataset archives or backup copies
print("\nDATASET BACKUPS")
print("-" * 70)

search_roots = [
    Path("/content"),
    Path("/content/drive/MyDrive/UA_FloodNet")
]

for root in search_roots:
    if root.exists():
        for p in root.iterdir():
            if any(
                word in p.name.lower()
                for word in ["ombria", "dataset", "floodnet"]
            ):
                print(p)

# 3. Inspect the restored training script
script = Path(
    "/content/UA_FloodNet/ablations_code/train_ablation.py"
)

print("\nTRAINING SCRIPT")
print("-" * 70)

if script.exists():
    for i, line in enumerate(
        script.read_text().splitlines(), start=1
    ):
        print(f"{i:3d} | {line}")
else:
    print("Training script missing.")

In [ ]:
from pathlib import Path
import zipfile

print("STEP 50: SEARCHING FOR ORIGINAL OMBRIA DATA")
print("=" * 70)

roots = [
    Path("/content"),
    Path("/content/drive/MyDrive")
]

keywords = [
    "ombria",
    "dataset",
    "floodnet",
    "train",
    "validation",
    "split"
]

found = []

for root in roots:
    if not root.exists():
        continue

    print(f"\nSearching: {root}")

    for path in root.rglob("*"):
        if not path.is_file():
            continue

        name = path.name.lower()

        if (
            path.suffix.lower() == ".npz"
            or (
                path.suffix.lower() in [".zip", ".tar", ".gz"]
                and any(word in name for word in keywords)
            )
            or (
                path.suffix.lower() in [".csv", ".json", ".txt"]
                and any(word in name for word in ["split", "manifest"])
            )
        ):
            found.append(path)

print("\nMATCHING FILES")
print("=" * 70)

print("Total matches:", len(found))

for path in found[:80]:
    print(path)

if len(found) > 80:
    print(f"... and {len(found)-80} additional matches")

print("\nZIP ARCHIVE INSPECTION")
print("=" * 70)

for path in found:
    if path.suffix.lower() != ".zip":
        continue

    try:
        with zipfile.ZipFile(path) as z:
            names = z.namelist()
            npz_count = sum(
                name.lower().endswith(".npz")
                for name in names
            )

            print(f"\nArchive: {path.name}")
            print("NPZ files:", npz_count)
            print("Entries:", len(names))
            print("First entries:", names[:8])

    except (zipfile.BadZipFile, OSError) as e:
        print(f"Could not inspect {path}: {e}")

print("\nSEARCH COMPLETE")

In [ ]:
from pathlib import Path
from collections import Counter
import zipfile

drive_root = Path("/content/drive/MyDrive")

print("STEP 51: TARGETED DATASET RECOVERY")
print("=" * 70)

# 1. Find actual NPZ files
npz_files = list(drive_root.rglob("*.npz"))

print("\nTOTAL NPZ FILES ON DRIVE:", len(npz_files))

if npz_files:
    folders = Counter(
        str(p.parent) for p in npz_files
    )

    print("\nNPZ FILES BY DIRECTORY")
    for folder, count in folders.most_common(30):
        print(f"{count:5d} | {folder}")

# 2. Identify potentially relevant archives
archives = [
    p for p in drive_root.rglob("*")
    if p.is_file()
    and p.suffix.lower() in [".zip", ".tar", ".gz", ".7z"]
    and any(
        word in str(p).lower()
        for word in [
            "ombria",
            "ua_floodnet",
            "ua-floodnet",
            "flood_dataset"
        ]
    )
]

print("\nPOTENTIAL DATASET ARCHIVES")
print("-" * 70)

for archive in archives:
    print(archive)

    if archive.suffix.lower() == ".zip":
        try:
            with zipfile.ZipFile(archive) as z:
                npz_count = sum(
                    n.lower().endswith(".npz")
                    for n in z.namelist()
                )
                print("  NPZ entries:", npz_count)
        except Exception as e:
            print("  Archive inspection failed:", str(e))

# 3. Inspect existing UA-FloodNet Drive folders
project_root = drive_root / "UA_FloodNet"

print("\nUA-FLOODNET DRIVE CONTENTS")
print("-" * 70)

if project_root.exists():
    for p in sorted(project_root.rglob("*")):
        if p.is_file():
            print(
                p.relative_to(project_root),
                "|",
                round(p.stat().st_size / 1024**2, 2),
                "MB"
            )
else:
    print("UA_FloodNet project folder not found.")

print("\nSTEP 51 COMPLETED")

In [ ]:
from pathlib import Path
import requests
import hashlib
import zipfile

root = Path("/content/OMBRIA")
root.mkdir(parents=True, exist_ok=True)

archive = root / "OMBRIA-master.zip"
url = "https://zenodo.org/records/6777870/files/OMBRIA-master.zip?download=1"

# Download only when necessary.
if not archive.exists():
    with requests.get(url, stream=True, timeout=60) as response:
        response.raise_for_status()
        with open(archive, "wb") as f:
            for chunk in response.iter_content(chunk_size=1024*1024):
                if chunk:
                    f.write(chunk)

# Verify official Zenodo MD5 checksum.
md5 = hashlib.md5()
with open(archive, "rb") as f:
    for chunk in iter(lambda: f.read(1024*1024), b""):
        md5.update(chunk)

expected = "3c4cd27b9c869bb32b30c7f10fecced3"
actual = md5.hexdigest()

print("Archive size (MB):", round(archive.stat().st_size / 1e6, 2))
print("MD5 verified:", actual == expected)

if actual != expected:
    raise RuntimeError("Checksum mismatch. Do not extract this archive.")

with zipfile.ZipFile(archive) as z:
    destination = root.resolve()
    for member in z.infolist():
        target = (destination / member.filename).resolve()
        if not target.is_relative_to(destination):
            raise ValueError(f"Unsafe archive entry: {member.filename}")
    z.extractall(root)

print("\nDATASET RESTORED")
print("=" * 60)

for p in sorted(root.iterdir()):
    print(p.name, "Directory:", p.is_dir())

print("\nSAMPLE SOURCE FOLDERS")
for p in list(root.rglob("BEFORE"))[:10]:
    print(p)

In [ ]:
from pathlib import Path
from collections import Counter
from PIL import Image
import numpy as np

root = Path("/content/OMBRIA/OMBRIA-master")

print("STEP 53: OMBRIA DATASET STRUCTURE VERIFICATION")
print("=" * 70)

for sensor in ["OmbriaS1", "OmbriaS2"]:
    for split in ["train", "test"]:
        print(f"\n{sensor.upper()} — {split.upper()}")
        print("-" * 55)

        for category in ["BEFORE", "AFTER", "MASK"]:
            folder = root / sensor / split / category

            if not folder.exists():
                print(f"{category}: MISSING")
                continue

            images = sorted(
                p for p in folder.iterdir()
                if p.suffix.lower() in [
                    ".png", ".jpg", ".jpeg", ".tif", ".tiff"
                ]
            )

            print(f"{category}: {len(images)} images")

            if images:
                example = images[0]
                with Image.open(example) as im:
                    arr = np.asarray(im)

                print("  Example filename:", example.name)
                print("  Image shape:", arr.shape)
                print("  Data type:", arr.dtype)
                print("  Value range:", arr.min(), arr.max())

# Check matching filenames across all categories.
print("\nPAIRING VERIFICATION")
print("=" * 70)

for split in ["train", "test"]:
    groups = {}

    for sensor in ["OmbriaS1", "OmbriaS2"]:
        for category in ["BEFORE", "AFTER", "MASK"]:
            folder = root / sensor / split / category

            if folder.exists():
                groups[f"{sensor}/{category}"] = {
                    p.stem for p in folder.iterdir()
                    if p.is_file()
                }

    if not groups:
        continue

    common = set.intersection(*groups.values())
    union = set.union(*groups.values())

    print(f"\n{split.upper()}")
    print("Common filenames:", len(common))
    print("Total unique filenames:", len(union))
    print("All groups match:", len(common) == len(union))

    for group, identifiers in groups.items():
        missing = union - identifiers
        print(
            f"{group}: {len(identifiers)} files, "
            f"{len(missing)} identifiers missing"
        )

print("\nStep 53 completed.")

In [ ]:
from pathlib import Path
import re
import numpy as np
from collections import Counter

root = Path("/content/OMBRIA/OMBRIA-master")

def extract_id(filename):
    match = re.search(r"_(\d+)\.[^.]+$", filename)
    if not match:
        raise ValueError(f"Cannot extract identifier: {filename}")
    return match.group(1)

def collect_ids(split):
    groups = {}

    for sensor in ["OmbriaS1", "OmbriaS2"]:
        for category in ["BEFORE", "AFTER", "MASK"]:

            folder = root / sensor / split / category

            files = [
                p for p in folder.iterdir()
                if p.suffix.lower() == ".png"
            ]

            ids = [extract_id(p.name) for p in files]

            assert len(ids) == len(set(ids)), (
                f"Duplicate identifiers in {folder}"
            )

            groups[f"{sensor}/{category}"] = set(ids)

    return groups

print("STEP 54: NUMERICAL PAIRING VERIFICATION")
print("=" * 65)

for split in ["train", "test"]:
    groups = collect_ids(split)

    common = set.intersection(*groups.values())
    union = set.union(*groups.values())

    print(f"\n{split.upper()}")
    print("Common identifiers:", len(common))
    print("Unique identifiers:", len(union))
    print("Complete pairing:", common == union)

    for name, ids in groups.items():
        print(f"{name}: {len(ids)}")

    if common != union:
        print("Unmatched identifiers:", sorted(union-common)[:20])

train_ids = sorted(
    set.intersection(*collect_ids("train").values())
)

test_ids = sorted(
    set.intersection(*collect_ids("test").values())
)

print("\nSOURCE SPLIT VERIFICATION")
print("=" * 65)

print("Official training pairs:", len(train_ids))
print("Official test pairs:", len(test_ids))

print(
    "Train/test identifier overlap:",
    len(set(train_ids) & set(test_ids))
)

assert len(train_ids) == 624
assert len(test_ids) == 70
assert set(train_ids).isdisjoint(test_ids)

print("\nCANDIDATE 499/125 RANDOM SPLIT")
print("-" * 65)

rng = np.random.default_rng(42)
shuffled_ids = rng.permutation(train_ids)

candidate_train = sorted(shuffled_ids[:499])
candidate_val = sorted(shuffled_ids[499:])

print("Candidate training pairs:", len(candidate_train))
print("Candidate validation pairs:", len(candidate_val))

print("First 15 candidate validation identifiers:")
print(candidate_val[:15])

print("\nStep 54 completed.")

In [ ]:
from pathlib import Path
from PIL import Image
import numpy as np
import pandas as pd
import json
import hashlib

source = Path("/content/OMBRIA/OMBRIA-master")
output = Path("/content/UA_FloodNet/dataset")
backup = Path("/content/drive/MyDrive/UA_FloodNet/dataset_backup")

output.mkdir(parents=True, exist_ok=True)
backup.mkdir(parents=True, exist_ok=True)

# Reconstruct candidate split deterministically.
train_ids = sorted(
    set.intersection(*collect_ids("train").values())
)
test_ids = sorted(
    set.intersection(*collect_ids("test").values())
)

rng = np.random.default_rng(42)
shuffled = rng.permutation(train_ids)

split_ids = {
    "train": sorted(shuffled[:499]),
    "val": sorted(shuffled[499:]),
    "test": sorted(test_ids)
}

def load_image(path):
    if not path.is_file():
        raise FileNotFoundError(path)
    return np.asarray(Image.open(path).convert(
        "L" if "S1_" in path.name or "mask_" in path.name.lower()
        else "RGB"
    ))

def convert_sample(identifier, source_split):
    s1 = source / "OmbriaS1" / source_split
    s2 = source / "OmbriaS2" / source_split

    sar_pre = load_image(
        s1 / "BEFORE" / f"S1_before_{identifier}.png"
    )
    sar_post = load_image(
        s1 / "AFTER" / f"S1_after_{identifier}.png"
    )
    ms_pre = load_image(
        s2 / "BEFORE" / f"S2_before_{identifier}.png"
    )
    ms_post = load_image(
        s2 / "AFTER" / f"S2_after_{identifier}.png"
    )

    mask_s1 = load_image(
        s1 / "MASK" / f"S1_mask_{identifier}.png"
    )
    mask_s2 = load_image(
        s2 / "MASK" / f"S2_mask_{identifier}.png"
    )

    assert sar_pre.shape == sar_post.shape == (256, 256)
    assert ms_pre.shape == ms_post.shape == (256, 256, 3)
    assert mask_s1.shape == mask_s2.shape == (256, 256)
    assert np.array_equal(mask_s1, mask_s2), (
        f"Mask mismatch: {identifier}"
    )

    return {
        "sar_pre": (sar_pre.astype(np.float32) / 255.0)[None],
        "sar_post": (sar_post.astype(np.float32) / 255.0)[None],
        "ms_pre": (ms_pre.astype(np.float32) / 255.0).transpose(2, 0, 1),
        "ms_post": (ms_post.astype(np.float32) / 255.0).transpose(2, 0, 1),
        "mask": (mask_s1 > 127).astype(np.float32)[None]
    }

manifest = []

for split, identifiers in split_ids.items():
    folder = output / split
    folder.mkdir(parents=True, exist_ok=True)

    original_split = "test" if split == "test" else "train"

    for identifier in identifiers:
        arrays = convert_sample(identifier, original_split)

        filename = f"{identifier}.npz"
        np.savez_compressed(folder / filename, **arrays)

        manifest.append({
            "identifier": identifier,
            "split": split,
            "source_split": original_split,
            "filename": filename
        })

    print(f"{split.upper()}: {len(list(folder.glob('*.npz')))} files")

manifest_df = pd.DataFrame(manifest)

assert manifest_df.identifier.nunique() == 694
assert len(manifest_df) == 694

manifest_path = backup / "candidate_split_seed42.csv"
manifest_df.to_csv(manifest_path, index=False)

metadata = {
    "dataset": "OMBRIA",
    "source_record": "Zenodo 6777870",
    "split_status": "candidate_not_verified_against_original_v1",
    "split_method": "numpy.default_rng(42).permutation(sorted_train_ids)",
    "train": 499,
    "val": 125,
    "test": 70,
    "normalization": "uint8 divided by 255",
    "mask": "binary, pixel value greater than 127",
    "note": "Original V1 train-validation assignment not established"
}

(backup / "conversion_metadata.json").write_text(
    json.dumps(metadata, indent=2)
)

print("\nSTEP 55: DATASET RECONSTRUCTION")
print("=" * 65)
print("Total NPZ files:", sum(
    len(list((output / s).glob("*.npz")))
    for s in split_ids
))
print("Unique identifiers:", manifest_df.identifier.nunique())
print("Manifest:", manifest_path)

example = np.load(next((output / "val").glob("*.npz")))

print("\nEXAMPLE NPZ")
for key in example.files:
    arr = example[key]
    print(
        key,
        arr.shape,
        arr.dtype,
        f"range=({arr.min():.3f}, {arr.max():.3f})"
    )

print("\nStep 55 completed.")

In [ ]:
import sys
import shutil
import torch
import numpy as np
from pathlib import Path
from torch.utils.data import DataLoader

sys.path.insert(
    0, "/content/UA_FloodNet/ablations_code"
)

from data import NPZFloodDataset
from ablation_model import UAFloodNet
from train_ablation import scores

data_root = Path("/content/UA_FloodNet/dataset")

print("STEP 56: DATASET COMPATIBILITY CHECK")
print("=" * 65)

# 1. Load reconstructed datasets
train_dataset = NPZFloodDataset(str(data_root / "train"))
val_dataset = NPZFloodDataset(str(data_root / "val"))
test_dataset = NPZFloodDataset(str(data_root / "test"))

print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Test samples:", len(test_dataset))

assert len(train_dataset) == 499
assert len(val_dataset) == 125
assert len(test_dataset) == 70

sample = train_dataset[0]

print("\nDATASET LOADER OUTPUT")
for key, value in sample.items():
    print(key, tuple(value.shape), value.dtype)

# 2. Restore existing full model
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

checkpoint = torch.load(
    "/content/drive/MyDrive/UA_FloodNet/"
    "ablations/full_seed42.pt",
    map_location="cpu",
    weights_only=True
)

model = UAFloodNet(
    width=checkpoint["width"],
    use_alignment=checkpoint["use_alignment"],
    use_reliability=checkpoint["use_reliability"]
).to(device)

model.load_state_dict(checkpoint["model"])
model.eval()

# 3. Evaluate on candidate validation split
loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=0
)

print("\nCANDIDATE VALIDATION PERFORMANCE")
print("-" * 65)

metrics = scores(model, loader, device)

for key, value in metrics.items():
    print(f"{key}: {value:.6f}")

print("\nORIGINAL CHECKPOINT VALIDATION METRICS")
print("-" * 65)

for key, value in checkpoint["metrics"].items():
    print(f"{key}: {value:.6f}")

print("\nVALIDATION METRIC DIFFERENCES")
print("-" * 65)

for key in metrics:
    if key in checkpoint["metrics"]:
        difference = metrics[key] - checkpoint["metrics"][key]
        print(f"{key}: {difference:+.6f}")

# 4. Create permanent compressed backup
backup_dir = Path(
    "/content/drive/MyDrive/UA_FloodNet/dataset_backup"
)

backup_dir.mkdir(parents=True, exist_ok=True)

archive_base = backup_dir / "OMBRIA_NPZ_candidate_seed42"

archive_path = Path(
    shutil.make_archive(
        str(archive_base),
        "zip",
        root_dir=str(data_root)
    )
)

print("\nDATASET BACKUP")
print("-" * 65)

print("Archive:", archive_path)
print(
    "Archive size:",
    round(archive_path.stat().st_size / 1024**2, 2),
    "MB"
)

print("\nSTEP 56 COMPLETED")

In [ ]:
from pathlib import Path

source = Path(
    "/content/UA_FloodNet/ablations_code/train_ablation.py"
)
target = Path(
    "/content/UA_FloodNet/ablations_code/train_v4.py"
)

code = source.read_text()

# Add augmentation controls.
old = "p.add_argument('--flow-weight',type=float,default=.01);a=p.parse_args()"
new = """p.add_argument('--flow-weight',type=float,default=.01)
    p.add_argument('--shift-prob',type=float,default=0.0)
    p.add_argument('--max-shift',type=int,default=4)
    a=p.parse_args()
    if not 0 <= a.shift_prob <= 1:
        raise ValueError('shift-prob must be between 0 and 1')
    if a.max_shift < 0:
        raise ValueError('max-shift must be nonnegative')"""

assert old in code, "Training argument insertion point not found"
code = code.replace(old, new, 1)

augmentation = '''
def translate_zero(x, dx, dy):
    _, _, h, w = x.shape
    result = torch.zeros_like(x)

    y0, y1 = max(0, dy), min(h, h + dy)
    x0, x1 = max(0, dx), min(w, w + dx)

    if y1 > y0 and x1 > x0:
        result[:, :, y0:y1, x0:x1] = x[
            :,
            :,
            y0-dy:y1-dy,
            x0-dx:x1-dx
        ]
    return result


def augment_misregistration(batch, probability, max_shift):
    """Independent paired translations for each sample."""
    if probability <= 0 or max_shift == 0:
        return batch

    pre = batch["ms_pre"]
    post = batch["ms_post"]

    pre_aug = []
    post_aug = []

    for i in range(pre.shape[0]):
        if torch.rand(()).item() < probability:
            dx = int(torch.randint(
                -max_shift, max_shift + 1, ()
            ).item())
            dy = int(torch.randint(
                -max_shift, max_shift + 1, ()
            ).item())
        else:
            dx, dy = 0, 0

        pre_aug.append(
            translate_zero(pre[i:i+1], dx, dy)
        )
        post_aug.append(
            translate_zero(post[i:i+1], dx, dy)
        )

    result = dict(batch)
    result["ms_pre"] = torch.cat(pre_aug, dim=0)
    result["ms_post"] = torch.cat(post_aug, dim=0)
    return result

'''

anchor = "\ndef main():"
assert anchor in code, "Main function insertion point not found"
code = code.replace(anchor, "\n" + augmentation + anchor, 1)

old_batch = "batch={k:v.to(device) for k,v in batch.items()}\n            opt.zero_grad"
new_batch = """batch={k:v.to(device) for k,v in batch.items()}
            batch=augment_misregistration(
                batch, a.shift_prob, a.max_shift
            )
            opt.zero_grad"""

assert old_batch in code, "Training batch insertion point not found"
code = code.replace(old_batch, new_batch, 1)

old_save = "'seed':a.seed},a.out)"
new_save = (
    "'seed':a.seed,"
    "'shift_prob':a.shift_prob,"
    "'max_shift':a.max_shift},a.out)"
)

assert old_save in code, "Checkpoint insertion point not found"
code = code.replace(old_save, new_save, 1)

target.write_text(code)

import py_compile
py_compile.compile(str(target), doraise=True)

print("STEP 57: V4 SCRIPT PREPARED")
print("=" * 60)
print("New script:", target)
print("Syntax verification: PASSED")
print("Original train_ablation.py: UNCHANGED")

In [ ]:
%cd /content/UA_FloodNet/ablations_code

!python train_v4.py --help

In [ ]:
import sys
import torch

sys.path.insert(
    0, "/content/UA_FloodNet/ablations_code"
)

from train_v4 import augment_misregistration

batch = {
    "sar_pre": torch.rand(3, 1, 256, 256),
    "sar_post": torch.rand(3, 1, 256, 256),
    "ms_pre": torch.rand(3, 3, 256, 256),
    "ms_post": torch.rand(3, 3, 256, 256),
    "mask": torch.randint(
        0, 2, (3, 1, 256, 256)
    ).float()
}

torch.manual_seed(42)

augmented = augment_misregistration(
    batch,
    probability=1.0,
    max_shift=4
)

print("STEP 57: AUGMENTATION SMOKE TEST")
print("=" * 60)

for key in batch:
    print(
        key,
        tuple(augmented[key].shape),
        "changed:",
        not torch.equal(batch[key], augmented[key])
    )

assert torch.equal(batch["sar_pre"], augmented["sar_pre"])
assert torch.equal(batch["sar_post"], augmented["sar_post"])
assert torch.equal(batch["mask"], augmented["mask"])

print("\nSmoke test passed.")

In [ ]:
%cd /content/UA_FloodNet/ablations_code

# V1 control: no misregistration augmentation
!python train_v4.py \
    --train /content/UA_FloodNet/dataset/train \
    --val /content/UA_FloodNet/dataset/val \
    --epochs 5 \
    --batch-size 4 \
    --lr 0.001 \
    --width 24 \
    --variant full \
    --seed 42 \
    --shift-prob 0.0 \
    --max-shift 4 \
    --out /content/drive/MyDrive/UA_FloodNet/v4/pilots/control_seed42_5ep.pt

# V4: misregistration-augmented training
!python train_v4.py \
    --train /content/UA_FloodNet/dataset/train \
    --val /content/UA_FloodNet/dataset/val \
    --epochs 5 \
    --batch-size 4 \
    --lr 0.001 \
    --width 24 \
    --variant full \
    --seed 42 \
    --shift-prob 0.75 \
    --max-shift 4 \
    --out /content/drive/MyDrive/UA_FloodNet/v4/pilots/augmented_seed42_5ep.pt

In [ ]:
%cd /content/UA_FloodNet/ablations_code

# Preserve the exact V4 training implementation on Google Drive.
!mkdir -p /content/drive/MyDrive/UA_FloodNet/v4/code
!cp train_v4.py /content/drive/MyDrive/UA_FloodNet/v4/code/train_v4.py

# Experiment A: control without augmentation
!python train_v4.py \
    --train /content/UA_FloodNet/dataset/train \
    --val /content/UA_FloodNet/dataset/val \
    --epochs 50 \
    --batch-size 4 \
    --lr 0.001 \
    --width 24 \
    --variant full \
    --seed 42 \
    --shift-prob 0.0 \
    --max-shift 4 \
    --out /content/drive/MyDrive/UA_FloodNet/v4/full/control_seed42_50ep.pt

In [ ]:
%cd /content/UA_FloodNet/ablations_code

# Experiment B: misregistration-augmented training
!python train_v4.py \
    --train /content/UA_FloodNet/dataset/train \
    --val /content/UA_FloodNet/dataset/val \
    --epochs 50 \
    --batch-size 4 \
    --lr 0.001 \
    --width 24 \
    --variant full \
    --seed 42 \
    --shift-prob 0.75 \
    --max-shift 4 \
    --out /content/drive/MyDrive/UA_FloodNet/v4/full/augmented_seed42_50ep.pt

In [ ]:
from google.colab import drive
from pathlib import Path
import zipfile
import shutil

# 1. Mount Google Drive
drive.mount("/content/drive")

project = Path("/content/UA_FloodNet")
code_dir = project / "ablations_code"
data_dir = project / "dataset"

code_dir.mkdir(parents=True, exist_ok=True)
data_dir.mkdir(parents=True, exist_ok=True)

# 2. Restore training code
archive = Path("/content/UA_FloodNet_Ablations.zip")

if archive.exists():
    with zipfile.ZipFile(archive) as z:
        z.extractall(code_dir)
    print("Ablation code restored.")
else:
    print("Ablation ZIP missing from runtime.")

# 3. Restore V4 training script from Drive
v4_source = Path(
    "/content/drive/MyDrive/UA_FloodNet/v4/code/train_v4.py"
)

if v4_source.exists():
    shutil.copy2(
        v4_source,
        code_dir / "train_v4.py"
    )
    print("V4 training script restored.")
else:
    print("V4 script backup not found.")

# 4. Restore reconstructed dataset
dataset_zip = Path(
    "/content/drive/MyDrive/UA_FloodNet/"
    "dataset_backup/OMBRIA_NPZ_candidate_seed42.zip"
)

if dataset_zip.exists():
    with zipfile.ZipFile(dataset_zip) as z:
        z.extractall(data_dir)
    print("OMBRIA dataset restored.")
else:
    print("Dataset backup not found.")

# 5. Verify full training checkpoints
checkpoints = {
    "Control 50 epochs":
        "/content/drive/MyDrive/UA_FloodNet/v4/full/control_seed42_50ep.pt",
    "Augmented 50 epochs":
        "/content/drive/MyDrive/UA_FloodNet/v4/full/augmented_seed42_50ep.pt"
}

print("\nCHECKPOINT VERIFICATION")
for name, location in checkpoints.items():
    p = Path(location)
    print(name, "FOUND" if p.exists() else "NOT FOUND")

print("\nRESTORATION VERIFICATION")
print("Training script:", (code_dir / "train_v4.py").exists())
print("Training samples:", len(list((data_dir / "train").glob("*.npz"))))
print("Validation samples:", len(list((data_dir / "val").glob("*.npz"))))
print("Test samples:", len(list((data_dir / "test").glob("*.npz"))))

In [ ]:
import torch
from pathlib import Path

models = {
    "V1 control": (
        "/content/drive/MyDrive/UA_FloodNet/"
        "v4/full/control_seed42_50ep.pt"
    ),
    "V4 augmented": (
        "/content/drive/MyDrive/UA_FloodNet/"
        "v4/full/augmented_seed42_50ep.pt"
    )
}

print("STEP 59B: FULL TRAINING CHECKPOINT VERIFICATION")
print("=" * 70)

for name, filename in models.items():

    path = Path(filename)

    print(f"\n{name.upper()}")
    print("-" * 55)
    print("Checkpoint exists:", path.exists())

    if not path.exists():
        continue

    checkpoint = torch.load(
        path,
        map_location="cpu",
        weights_only=True
    )

    print("File size:", round(path.stat().st_size / 1024**2, 2), "MB")
    print("Best saved epoch:", checkpoint.get("epoch"))
    print("Architecture:", checkpoint.get("variant"))
    print("Width:", checkpoint.get("width"))
    print("Random seed:", checkpoint.get("seed"))
    print("Shift probability:", checkpoint.get("shift_prob"))
    print("Maximum shift:", checkpoint.get("max_shift"))

    print("\nBEST SAVED VALIDATION METRICS")

    metrics = checkpoint.get("metrics", {})

    for key, value in metrics.items():
        print(f"{key}: {value:.6f}")

print("\nCHECKPOINT INSPECTION COMPLETED")

In [ ]:
from google.colab import files

uploaded = files.upload()

print("\nUploaded files:")
for filename in uploaded:
    print(filename)

In [ ]:
from pathlib import Path
import zipfile
import sys

# Find uploaded ZIP, regardless of current directory
matches = list(Path("/content").rglob("UA_FloodNet_Ablations.zip"))

assert matches, "Ablation ZIP not found. Check Colab Files."

archive = matches[0]
destination = Path("/content/UA_FloodNet/ablations_code")
destination.mkdir(parents=True, exist_ok=True)

print("Archive found:", archive)

with zipfile.ZipFile(archive) as z:
    for filename in ["ablation_model.py", "data.py"]:
        target = destination / filename
        target.write_bytes(z.read(filename))
        print("Restored:", target)

sys.path.insert(0, str(destination))

from ablation_model import UAFloodNet
from data import NPZFloodDataset

print("\nSUCCESS: Model dependencies restored.")

In [ ]:
import shutil
from pathlib import Path

source = Path("/content/UA_FloodNet/ablations_code")
backup = Path("/content/drive/MyDrive/UA_FloodNet/v4/code")

backup.mkdir(parents=True, exist_ok=True)

for filename in ["ablation_model.py", "data.py", "train_v4.py"]:
    shutil.copy2(source / filename, backup / filename)
    print("Backed up:", filename)

print("\nSUCCESS: All essential code saved to Google Drive.")

In [ ]:
import sys
from pathlib import Path
import torch
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader

sys.path.insert(0, "/content/UA_FloodNet/ablations_code")

from ablation_model import UAFloodNet
from data import NPZFloodDataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

checkpoint_dir = Path(
    "/content/drive/MyDrive/UA_FloodNet/v4/full"
)

models = {}

for name, filename in {
    "V1_control": "control_seed42_50ep.pt",
    "V4_augmented": "augmented_seed42_50ep.pt"
}.items():

    checkpoint = torch.load(
        checkpoint_dir / filename,
        map_location="cpu",
        weights_only=True
    )

    model = UAFloodNet(
        width=checkpoint["width"],
        use_alignment=checkpoint["use_alignment"],
        use_reliability=checkpoint["use_reliability"]
    ).to(device)

    model.load_state_dict(checkpoint["model"])
    model.eval()
    models[name] = model

print("Loaded models:", list(models))
print("Device:", device)

def translate_zero(x, dx, dy):
    _, _, h, w = x.shape
    out = torch.zeros_like(x)

    y0, y1 = max(0, dy), min(h, h + dy)
    x0, x1 = max(0, dx), min(w, w + dx)

    if y1 > y0 and x1 > x0:
        out[:, :, y0:y1, x0:x1] = x[
            :, :,
            y0-dy:y1-dy,
            x0-dx:x1-dx
        ]

    return out

dataset = NPZFloodDataset(
    "/content/UA_FloodNet/dataset/val"
)

loader = DataLoader(
    dataset,
    batch_size=4,
    shuffle=False,
    num_workers=0
)

shifts = [
    (0, 0),
    (1, 0), (-1, 0),
    (2, 0), (-2, 0),
    (4, 0), (-4, 0),
    (8, 0), (-8, 0),
    (0, 2), (0, -2),
    (0, 4), (0, -4)
]

records = []
margin = 20

with torch.inference_mode():

    for dx, dy in shifts:

        totals = {
            name: {"tp": 0, "fp": 0, "fn": 0}
            for name in models
        }

        for batch in loader:
            batch = {
                k: v.to(device)
                for k, v in batch.items()
            }

            shifted_pre = translate_zero(
                batch["ms_pre"], dx, dy
            )
            shifted_post = translate_zero(
                batch["ms_post"], dx, dy
            )

            gt = batch["mask"] >= 0.5

            valid = torch.zeros_like(gt, dtype=torch.bool)
            valid[:, :, margin:-margin, margin:-margin] = True

            for name, model in models.items():

                logits = model(
                    batch["sar_pre"],
                    batch["sar_post"],
                    shifted_pre,
                    shifted_post
                )["logits"]

                prediction = torch.sigmoid(logits) >= 0.5

                totals[name]["tp"] += int(
                    (prediction & gt & valid).sum().item()
                )
                totals[name]["fp"] += int(
                    (prediction & ~gt & valid).sum().item()
                )
                totals[name]["fn"] += int(
                    (~prediction & gt & valid).sum().item()
                )

        for name, c in totals.items():
            tp, fp, fn = c["tp"], c["fp"], c["fn"]

            records.append({
                "model": name,
                "dx": dx,
                "dy": dy,
                "magnitude": float(np.hypot(dx, dy)),
                "IoU": tp / max(tp + fp + fn, 1),
                "F1": 2 * tp / max(2 * tp + fp + fn, 1)
            })

results60 = pd.DataFrame(records)

print("\nSTEP 60: CONTROL VS AUGMENTED ROBUSTNESS")
print("=" * 70)

print("\nIoU BY TRANSLATION")
print(
    results60.pivot_table(
        index=["dx", "dy"],
        columns="model",
        values="IoU"
    ).round(5).to_string()
)

print("\nROBUSTNESS SUMMARY")
for name, group in results60.groupby("model"):

    clean = group[
        (group.dx == 0) & (group.dy == 0)
    ].iloc[0]

    shifted = group[
        ~((group.dx == 0) & (group.dy == 0))
    ]

    print(f"\n{name}")
    print("Clean IoU:", round(clean.IoU, 5))
    print("Mean shifted IoU:", round(shifted.IoU.mean(), 5))
    print(
        "Mean IoU loss:",
        round(clean.IoU - shifted.IoU.mean(), 5)
    )
    print(
        "Worst shifted IoU:",
        round(shifted.IoU.min(), 5)
    )

save_dir = Path(
    "/content/drive/MyDrive/UA_FloodNet/v4/robustness"
)
save_dir.mkdir(parents=True, exist_ok=True)

results60.to_csv(
    save_dir / "step60_control_vs_augmented.csv",
    index=False
)

print("\nResults saved:", save_dir)

In [ ]:
%cd /content/UA_FloodNet/ablations_code

!mkdir -p /content/drive/MyDrive/UA_FloodNet/v4/light

In [ ]:
import subprocess
from pathlib import Path

out_dir = Path("/content/drive/MyDrive/UA_FloodNet/v4/light")
out_dir.mkdir(parents=True, exist_ok=True)

command = [
    "python", "train_v4.py",
    "--train", "/content/UA_FloodNet/dataset/train",
    "--val", "/content/UA_FloodNet/dataset/val",
    "--epochs", "50",
    "--batch-size", "4",
    "--lr", "0.001",
    "--width", "24",
    "--variant", "full",
    "--seed", "42",
    "--shift-prob", "0.25",
    "--max-shift", "2",
    "--out", str(out_dir / "light_seed42_50ep.pt")
]

log_path = out_dir / "light_seed42_training.log"

with log_path.open("w") as log:
    result = subprocess.run(
        command,
        stdout=log,
        stderr=subprocess.STDOUT,
        text=True
    )

print("Training exit code:", result.returncode)
print("Training log:", log_path)
print("Checkpoint exists:",
      (out_dir / "light_seed42_50ep.pt").exists())

print("\nLAST 12 LOG LINES")
print("\n".join(log_path.read_text().splitlines()[-12:]))

if result.returncode != 0:
    print("Training failed or was interrupted. Inspect the saved log.")

In [ ]:
import sys
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from torch.utils.data import DataLoader

code_dir = Path("/content/UA_FloodNet/ablations_code")
sys.path.insert(0, str(code_dir))

from ablation_model import UAFloodNet
from data import NPZFloodDataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
drive_root = Path("/content/drive/MyDrive/UA_FloodNet")

checkpoint_paths = {
    "V1_control": drive_root / "v4/full/control_seed42_50ep.pt",
    "V4_strong": drive_root / "v4/full/augmented_seed42_50ep.pt",
    "V4_light": drive_root / "v4/light/light_seed42_50ep.pt"
}

models = {}

for name, path in checkpoint_paths.items():
    assert path.exists(), f"Missing checkpoint: {path}"

    ckpt = torch.load(path, map_location="cpu", weights_only=True)

    model = UAFloodNet(
        width=ckpt["width"],
        use_alignment=ckpt["use_alignment"],
        use_reliability=ckpt["use_reliability"]
    ).to(device)

    model.load_state_dict(ckpt["model"])
    model.eval()
    models[name] = model

print("Loaded models:", list(models))
print("Device:", device)

def translate_zero(x, dx, dy):
    _, _, h, w = x.shape
    out = torch.zeros_like(x)

    y0, y1 = max(0, dy), min(h, h + dy)
    x0, x1 = max(0, dx), min(w, w + dx)

    if y1 > y0 and x1 > x0:
        out[:, :, y0:y1, x0:x1] = x[
            :, :, y0-dy:y1-dy, x0-dx:x1-dx
        ]

    return out

val_dir = "/content/UA_FloodNet/dataset/val"
dataset = NPZFloodDataset(val_dir)
assert len(dataset) == 125

loader = DataLoader(
    dataset, batch_size=4, shuffle=False, num_workers=0
)

shifts = [
    (0, 0),
    (-1, 0), (1, 0),
    (-2, 0), (2, 0),
    (-4, 0), (4, 0),
    (-8, 0), (8, 0),
    (0, -2), (0, 2),
    (0, -4), (0, 4)
]

records = []
margin = 20

with torch.inference_mode():
    for dx, dy in shifts:
        totals = {
            name: {"tp": 0, "fp": 0, "fn": 0}
            for name in models
        }

        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}

            shifted_pre = translate_zero(batch["ms_pre"], dx, dy)
            shifted_post = translate_zero(batch["ms_post"], dx, dy)

            gt = batch["mask"] >= 0.5

            valid = torch.zeros_like(gt, dtype=torch.bool)
            valid[:, :, margin:-margin, margin:-margin] = True

            for name, model in models.items():
                logits = model(
                    batch["sar_pre"],
                    batch["sar_post"],
                    shifted_pre,
                    shifted_post
                )["logits"]

                pred = logits.sigmoid() >= 0.5

                totals[name]["tp"] += (
                    pred & gt & valid
                ).sum().item()

                totals[name]["fp"] += (
                    pred & ~gt & valid
                ).sum().item()

                totals[name]["fn"] += (
                    ~pred & gt & valid
                ).sum().item()

        for name, counts in totals.items():
            tp, fp, fn = (
                counts["tp"], counts["fp"], counts["fn"]
            )

            records.append({
                "model": name,
                "dx": dx,
                "dy": dy,
                "IoU": tp / max(tp + fp + fn, 1),
                "F1": 2 * tp / max(2 * tp + fp + fn, 1)
            })

results62 = pd.DataFrame(records)

print("\nSTEP 62: THREE-MODEL ROBUSTNESS COMPARISON")
print("=" * 72)

print("\nIoU BY TRANSLATION")
print(
    results62.pivot_table(
        index=["dx", "dy"],
        columns="model",
        values="IoU"
    ).round(5).to_string()
)

summary = []

for name, group in results62.groupby("model"):
    clean = group[
        (group.dx == 0) & (group.dy == 0)
    ].iloc[0]

    shifted = group[
        ~((group.dx == 0) & (group.dy == 0))
    ]

    summary.append({
        "model": name,
        "clean_iou": clean.IoU,
        "mean_shifted_iou": shifted.IoU.mean(),
        "mean_iou_loss": clean.IoU - shifted.IoU.mean(),
        "worst_shifted_iou": shifted.IoU.min(),
        "clean_f1": clean.F1,
        "mean_shifted_f1": shifted.F1.mean()
    })

summary62 = pd.DataFrame(summary).set_index("model")

print("\nROBUSTNESS SUMMARY")
print(summary62.round(5).to_string())

# Compare absolute robustness performance against V1.
control = summary62.loc["V1_control"]

print("\nDIFFERENCE FROM V1 CONTROL")
for name in ["V4_strong", "V4_light"]:
    row = summary62.loc[name]
    print(
        name,
        "clean IoU difference:",
        round(row.clean_iou - control.clean_iou, 5),
        "mean shifted IoU difference:",
        round(row.mean_shifted_iou - control.mean_shifted_iou, 5)
    )

save_dir = drive_root / "v4/robustness"
save_dir.mkdir(parents=True, exist_ok=True)

results62.to_csv(
    save_dir / "step62_three_model_results.csv",
    index=False
)

summary62.to_csv(
    save_dir / "step62_three_model_summary.csv"
)

print("\nStep 62 completed. Results saved to Google Drive.")